# Algorithm 18 From-Scratch KLDM-PPR

This notebook rebuilds Algorithm 18 around the **from-scratch KLDM-PPR fractional/velocity** formulation.

Active method:

1. optimize noisy `(F_t, V_t)` through `D_f`
2. evaluate a fixed **SVD Wyckoff normal constraint** on the clean estimate
3. renoise through KLDM's native forward kernel with `V0 = 0`
4. repeat the fixed-time PPR kernel

We also keep explicit **frame-problem diagnostics** in the notebook, because that is still one of the main places things can fail for graphs 3 and 5.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print(f'ROOT={ROOT}')


ROOT=/workspace


In [2]:
import os
import math
import random
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml


try:
    from scipy.optimize import linear_sum_assignment
except ImportError:
    linear_sum_assignment = None

from kldmPlus.algorithm10_casal_chart import _decode_lattice_matrix, _encode_lattice_matrix, _l_to_k
import kldmPlus.algorithm18_hard_fractional_kldm_ppr as algo18_backend
algo18_backend = importlib.reload(algo18_backend)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import (
    build_pyxtal_wyckoff_result,
    build_symmetry_frame_bridge,
    expand_wyckoff_template_torch,
    extract_wyckoff_templates,
    flatten_site_signature,
    map_standardized_structure_to_vanilla_frame,
    species_aware_torus_rmse,
    estimate_semantic_transport_for_reference_order,
    transport_standardized_frac_to_vanilla_frame_with_tau,
    transport_standardized_structure_to_vanilla_frame,
    transport_standardized_tangent_block_to_vanilla_frame,
)
from kldmPlus.symmetry.pcs_projection import _periodic_pairwise_distances

ALGORITHM18_DESCRIPTION = algo18_backend.ALGORITHM18_DESCRIPTION
ALGORITHM18_IS_FULL_PPR = algo18_backend.ALGORITHM18_IS_FULL_PPR
ALGORITHM18_MODE = algo18_backend.ALGORITHM18_MODE
ALGORITHM18_RELATION_TO_PPR = algo18_backend.ALGORITHM18_RELATION_TO_PPR
ALGORITHM18_SHORT_NAME = algo18_backend.ALGORITHM18_SHORT_NAME
Algorithm18Config = algo18_backend.Algorithm18Config
Algorithm18GraphState = algo18_backend.Algorithm18GraphState
FixedWyckoffFrame = algo18_backend.FixedWyckoffFrame
SVDWyckoffConstraint = algo18_backend.SVDWyckoffConstraint
SVDOrbitConstraint = algo18_backend.SVDOrbitConstraint
build_fixed_wyckoff_frame = algo18_backend.build_fixed_wyckoff_frame
build_oracle_diffcsppp_payload_from_structure = algo18_backend.build_oracle_diffcsppp_payload_from_structure
build_template_and_free_vars_from_diffcsppp_payload = algo18_backend.build_template_and_free_vars_from_diffcsppp_payload
build_template_and_free_vars_from_pyxtal = algo18_backend.build_template_and_free_vars_from_pyxtal
recover_template_free_vars_from_anchor_entries = algo18_backend.recover_template_free_vars_from_anchor_entries
denoiser_shape_and_finite_sanity = algo18_backend.denoiser_shape_and_finite_sanity
graph_center_velocity = algo18_backend.graph_center_velocity
kldm_clean_fractional_denoiser_Df = algo18_backend.kldm_clean_fractional_denoiser_Df
kldm_renoise_from_f0 = algo18_backend.kldm_renoise_from_f0
oracle_clean_denoiser_sign_check = algo18_backend.oracle_clean_denoiser_sign_check
ppr_kernel_repeated = algo18_backend.ppr_kernel_repeated
ppr_objective_gradient_sanity = algo18_backend.ppr_objective_gradient_sanity
ppr_project_step = algo18_backend.ppr_project_step
signed_wrap = algo18_backend.signed_wrap
torus_rmse = algo18_backend.torus_rmse
wrap01 = algo18_backend.wrap01
species_align_to_target_order = algo18_backend.species_align_to_target_order

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
PROFILE = os.environ.get('KLDM_WYCKOFF_PPR_PROFILE', 'laptop').strip().lower()


def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop


def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]


SAMPLE_SEED = int(profile_default('KLDM_WYCKOFF_PPR_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_WYCKOFF_PPR_GRAPH_IDS', '2,3,5', '2,3,5'))
DEBUG_PRINTS = bool(int(profile_default('KLDM_WYCKOFF_PPR_DEBUG', '1', '1')))
ALGO18_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_ALGO18_T_VALUES', '0.2,0.3', '0.2,0.3')).split(',') if v.strip())
ALGO18_M_VALUES = tuple(int(v.strip()) for v in str(profile_default('KLDM_ALGO18_M_VALUES', '0,1,2', '0,1,2')).split(',') if v.strip())
ALGO18_VARIANTS = tuple(v.strip() for v in str(profile_default('KLDM_ALGO18_VARIANTS', 'correctW,wrongSG', 'correctW,wrongSG')).split(',') if v.strip())
ALGO18_PROJ_STEPS = int(profile_default('KLDM_ALGO18_PROJ_STEPS', '6', '8'))
ALGO18_LR = float(profile_default('KLDM_ALGO18_LR', '2e-2', '2e-2'))
ALGO18_LAMBDA_NOISE = float(profile_default('KLDM_ALGO18_LAMBDA_NOISE', '1e-2', '1e-2'))
ALGO18_OPTIMIZE_VELOCITY = bool(int(profile_default('KLDM_ALGO18_OPTIMIZE_VELOCITY', '1', '1')))
ALGO18_DENOISER_VARIANT = str(profile_default('KLDM_ALGO18_DENOISER_VARIANT', 'minus', 'minus'))
ALGO18_C_SCI_EPS = float(profile_default('KLDM_ALGO18_C_SCI_EPS', '1e-4', '1e-4'))
ALGO18_FRAC_RMSE_TOL = float(profile_default('KLDM_ALGO18_FRAC_RMSE_TOL', '5e-3', '5e-3'))
MAX_TEMPLATES = int(profile_default('KLDM_WYCKOFF_PPR_MAX_TEMPLATES', '64', '64'))

print(f'profile={PROFILE} graphs={GRAPH_IDS} t_values={ALGO18_T_VALUES} M_values={ALGO18_M_VALUES} variants={ALGO18_VARIANTS}')
print(f'proj_steps={ALGO18_PROJ_STEPS} lr={ALGO18_LR} lambda_noise={ALGO18_LAMBDA_NOISE} optimize_velocity={ALGO18_OPTIMIZE_VELOCITY}')
print(f'backend_module={algo18_backend.__file__}')
print(ALGORITHM18_DESCRIPTION)
print(ALGORITHM18_RELATION_TO_PPR)


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
profile=laptop graphs=[2, 3, 5] t_values=(0.2, 0.3) M_values=(0, 1, 2) variants=('correctW', 'wrongSG')
proj_steps=6 lr=0.02 lambda_noise=0.01 optimize_velocity=True
backend_module=/workspace/src/kldmPlus/algorithm18_hard_fractional_kldm_ppr.py
From-scratch KLDM-compatible PPR for the fractional-coordinate/velocity branch: optimize xi_r, xi_v through D_f and a fixed SVD Wyckoff normal constraint, then renoise with KLDM's native forward kernel while keeping the lattice fixed.
Follows PPR's project-through-denoiser, renoise, and repeat structure at fixed t, adapted to KLDM's torus-plus-velocity state and simplified clean-velocity convention V0=0.


In [3]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
_ = runner._ensure_template_prior()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
if not available_graph_ids:
    raise RuntimeError(f'No requested graph ids are present. requested={GRAPH_IDS} available=1..{full_num_graphs}')
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.tolist()
print(f'loaded_graph_ids={available_graph_ids} ptr={ptr}')


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train

In [22]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int


@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    k: torch.Tensor
    h: torch.Tensor
    t: float
    dt: float
    graph_idx0: int


result_tables: dict[str, pd.DataFrame] = {}
correct_fixtures: dict[int, dict[str, Any]] = {}
wrong_fixtures: dict[int, dict[str, Any]] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_lattice=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def benchmark_cases() -> list[GraphCase]:
    selected = {int(g) for g in GRAPH_IDS}
    return [case for case in GRAPH_CASES if int(case.graph_id) in selected]


def l_to_k(l: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _l_to_k(
        l=l.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def ensure_l_feature(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        encoded = _encode_lattice_matrix(
            cell_matrix=flat.reshape(3, 3),
            num_atoms=int(num_atoms),
            lattice_transform=runner.lattice_transform,
        )
        return encoded.to(device=l.device, dtype=l.dtype).reshape(-1)
    return flat


def cell_from_l(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        return flat.reshape(3, 3).detach()
    return _decode_lattice_matrix(
        l=flat,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).detach()


def min_pair_distance_from_arrays(frac: torch.Tensor, l: torch.Tensor, num_atoms: int) -> float:
    cell = cell_from_l(l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    distances = _periodic_pairwise_distances(frac_coords=wrap01(frac), cell_matrix=cell)
    return float(distances.min().detach().item()) if distances.numel() else float('inf')


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    return (edge_index[:, mask] - start).detach().clone()


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    pred_l_feature = ensure_l_feature(pred_l, int(pred_f.shape[0]))
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l_feature,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'min_pair_distance': result.min_pair_distance,
        'metric_source': 'sample_evaluation.evaluate_csp_reconstruction',
    }


def algo18_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def algo18_state_from_noisy_gt(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED) -> KLDMState:
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    l0 = ensure_l_feature(case.gt_lattice, int(case.gt_frac.shape[0]))
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    return KLDMState(
        f=f_t.detach().clone(),
        v=v_t.detach().clone(),
        l=l0.detach().clone(),
        k=l_to_k(l0, case),
        h=case.atomic_numbers.detach().clone(),
        t=float(t_value),
        dt=0.0,
        graph_idx0=case.graph_idx0,
    )


def algo18_to_backend_state(state: KLDMState) -> Algorithm18GraphState:
    return Algorithm18GraphState(
        f=state.f.detach().clone(),
        v=state.v.detach().clone(),
        l=state.l.detach().clone(),
        h=state.h.detach().clone(),
        k=state.k.detach().clone(),
        t=float(state.t),
        dt=float(state.dt),
        graph_idx0=int(state.graph_idx0),
    )


def _semantic_transport_metadata(bridge) -> dict[str, Any]:
    return {
        'method': getattr(bridge, 'standardized_to_vanilla_method', None),
        'rmse': getattr(bridge, 'standardized_to_vanilla_rmse', None),
        'tau': None if getattr(bridge, 'standardized_to_vanilla_tau', None) is None else np.asarray(bridge.standardized_to_vanilla_tau, dtype=float).tolist(),
        'assignment': None if getattr(bridge, 'standardized_to_vanilla_assignment', None) is None else np.asarray(bridge.standardized_to_vanilla_assignment, dtype=int).tolist(),
    }



def _ordered_reference_transport_metadata(bridge, standardized_reference_frac: torch.Tensor, standardized_reference_atomic_numbers: torch.Tensor) -> dict[str, Any]:
    try:
        tau, assignment, rmse = estimate_semantic_transport_for_reference_order(
            standardized_reference_frac_coords=np.asarray(standardized_reference_frac.detach().cpu(), dtype=float),
            standardized_reference_atomic_numbers=np.asarray(standardized_reference_atomic_numbers.detach().cpu(), dtype=int),
            bridge=bridge,
        )
        return {
            'method': 'ordered_semantic_transport',
            'tau': np.asarray(tau, dtype=float).tolist(),
            'assignment': np.asarray(assignment, dtype=int).tolist(),
            'rmse': float(rmse),
        }
    except Exception as exc:
        return {
            'method': 'ordered_semantic_transport_failed',
            'error': f'{type(exc).__name__}: {exc}',
            'tau': None,
            'assignment': None,
            'rmse': None,
        }



def _standardized_template_to_vanilla_tensors(case: GraphCase, bridge, template, free_vars: torch.Tensor, *, reference_transport: dict[str, Any] | None = None) -> tuple[torch.Tensor, torch.Tensor, float | None, str | None]:
    expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars, wrap=True)
    transport_status = 'ordered_semantic_transport'
    try:
        if not reference_transport or reference_transport.get('tau') is None:
            raise RuntimeError('missing_ordered_reference_transport')
        frac_np = transport_standardized_frac_to_vanilla_frame_with_tau(
            standardized_frac_coords=np.asarray(expansion.frac_coords.detach().cpu(), dtype=float),
            bridge=bridge,
            tau=np.asarray(reference_transport['tau'], dtype=float),
        )
        frac = torch.as_tensor(frac_np, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        atomic_numbers = expansion.atomic_numbers.to(device=case.gt_frac.device, dtype=torch.long)
    except Exception:
        std_cell = torch.as_tensor(np.asarray(bridge.standardized_structure.lattice.matrix, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        std_l_feature = ensure_l_feature(std_cell.reshape(-1), int(expansion.frac_coords.shape[0]))
        standardized_pred = build_structure_from_sample(
            f=expansion.frac_coords,
            l=std_l_feature,
            a=expansion.atomic_numbers,
            lattice_transform=runner.lattice_transform,
        )
        transport_status = 'structure_matcher_fallback'
        vanilla_like = map_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_pred,
            vanilla_reference_structure=build_case_structure(case),
            symprec=1e-2,
            angle_tolerance=5.0,
            stol=0.5,
            ltol=0.3,
        )
        frac = torch.as_tensor(np.asarray(vanilla_like.frac_coords, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        atomic_numbers = torch.as_tensor(np.asarray(vanilla_like.atomic_numbers, dtype=int), device=case.gt_frac.device, dtype=torch.long)
    rmse, status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(frac.detach().cpu(), dtype=float),
        source_atomic_numbers=np.asarray(atomic_numbers.detach().cpu(), dtype=int),
        target_frac_coords=np.asarray(case.gt_frac.detach().cpu(), dtype=float),
        target_atomic_numbers=np.asarray(case.atomic_numbers.detach().cpu(), dtype=int),
    )
    status = transport_status if status == 'ok' else f'{transport_status}:{status}'
    return frac, atomic_numbers, rmse, status



def _bridge_standardized_to_vanilla_tensors(case: GraphCase, bridge, payload) -> tuple[torch.Tensor, torch.Tensor, float | None, str | None, dict[str, Any]]:
    standardized_gt_frac = torch.as_tensor(np.asarray(payload.expanded_frac_coords, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    standardized_gt_z = torch.as_tensor(np.asarray(payload.expanded_atomic_numbers, dtype=int), device=case.gt_frac.device, dtype=torch.long)
    ordered_transport = _ordered_reference_transport_metadata(bridge, standardized_gt_frac, standardized_gt_z)
    transport_status = str(ordered_transport.get('method'))
    try:
        if ordered_transport.get('tau') is None:
            raise RuntimeError('ordered_transport_missing_tau')
        frac_np = transport_standardized_frac_to_vanilla_frame_with_tau(
            standardized_frac_coords=np.asarray(standardized_gt_frac.detach().cpu(), dtype=float),
            bridge=bridge,
            tau=np.asarray(ordered_transport['tau'], dtype=float),
        )
        frac = torch.as_tensor(frac_np, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        atomic_numbers = standardized_gt_z.detach().clone()
    except Exception:
        transport_status = 'structure_matcher_bridge'
        vanilla_like = bridge.standardized_to_vanilla_structure
        frac = torch.as_tensor(np.asarray(vanilla_like.frac_coords, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        atomic_numbers = torch.as_tensor(np.asarray(vanilla_like.atomic_numbers, dtype=int), device=case.gt_frac.device, dtype=torch.long)
    rmse, status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(frac.detach().cpu(), dtype=float),
        source_atomic_numbers=np.asarray(atomic_numbers.detach().cpu(), dtype=int),
        target_frac_coords=np.asarray(case.gt_frac.detach().cpu(), dtype=float),
        target_atomic_numbers=np.asarray(case.atomic_numbers.detach().cpu(), dtype=int),
    )
    status = transport_status if status == 'ok' else f'{transport_status}:{status}'
    return frac, atomic_numbers, rmse, status, ordered_transport




def _hungarian_local(cost: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if linear_sum_assignment is not None:
        rows, cols = linear_sum_assignment(cost)
        return np.asarray(rows, dtype=int), np.asarray(cols, dtype=int)
    remaining_rows = list(range(cost.shape[0]))
    remaining_cols = list(range(cost.shape[1]))
    chosen_rows = []
    chosen_cols = []
    while remaining_rows:
        sub = cost[np.ix_(remaining_rows, remaining_cols)]
        flat = int(np.argmin(sub))
        ncols = sub.shape[1]
        r_pos = flat // ncols
        c_pos = flat % ncols
        chosen_rows.append(remaining_rows.pop(r_pos))
        chosen_cols.append(remaining_cols.pop(c_pos))
    order = np.argsort(np.asarray(chosen_rows))
    return np.asarray(chosen_rows, dtype=int)[order], np.asarray(chosen_cols, dtype=int)[order]


def _refine_assignment_within_orbits(frame: FixedWyckoffFrame, target_frac: torch.Tensor) -> tuple[torch.Tensor, list[dict[str, Any]]]:
    assignment = frame.assignment.detach().clone()
    debug = []
    target01 = wrap01(target_frac)
    for orbit in frame.constraint.orbits:
        n = int(orbit.target_indices.numel())
        if n <= 1:
            debug.append({'label': orbit.label, 'size': n, 'changed': False, 'rmse_before': 0.0, 'rmse_after': 0.0})
            continue
        template_idx = torch.arange(n, device=assignment.device, dtype=torch.long) + int(orbit.target_indices[0].numel() * 0)
        # orbit target indices correspond to a contiguous template block because frame.orbits were built in order
        start = int(orbit.target_indices.numel() * 0)
        # recover template block by matching label/site order from frame.reference_template_order against target indices block size
        # use the already stored orbit reference block in target-orbit order as source for local reassignment
        src = orbit.reference_template_block.detach().reshape(n, 3).cpu().numpy()
        dst_idx = orbit.target_indices.detach().cpu().numpy().astype(int)
        dst = target01[orbit.target_indices].detach().cpu().numpy()
        delta = src[:, None, :] - dst[None, :, :]
        delta = delta - np.round(delta)
        cost = np.sum(delta * delta, axis=-1)
        rows, cols = _hungarian_local(cost)
        rmse_before = float(np.sqrt(np.mean(np.sum((src - dst - np.round(src - dst))**2, axis=-1))))
        reassigned = dst_idx[cols[np.argsort(rows)]]
        rmse_after = float(np.sqrt(np.mean(cost[rows, cols]) / 3.0)) if cost.size else 0.0
        changed = bool(not np.array_equal(dst_idx, reassigned))
        assignment[orbit.target_indices.to(device=assignment.device, dtype=torch.long)] = torch.as_tensor(reassigned, device=assignment.device, dtype=torch.long)
        debug.append({'label': orbit.label, 'size': n, 'changed': changed, 'old_indices': dst_idx.tolist(), 'new_indices': reassigned.tolist(), 'rmse_before': rmse_before, 'rmse_after': rmse_after})
    return assignment, debug


def _build_repaired_frame(case: GraphCase, template, free_vars: torch.Tensor, base_frame: FixedWyckoffFrame) -> tuple[FixedWyckoffFrame, list[dict[str, Any]]]:
    repaired_assignment, repair_debug = _refine_assignment_within_orbits(base_frame, case.gt_frac)
    repaired_frame = build_fixed_wyckoff_frame(
        template=template,
        free_vars=free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        reference_frac_target_order=base_frame.reference_target_order,
        atomic_numbers_target_order=base_frame.atomic_numbers_target_order,
        assignment=repaired_assignment,
        reason=base_frame.reason + '_orbit_repaired',
        debug=DEBUG_PRINTS,
    )
    return repaired_frame, repair_debug


def _scan_best_global_tau(reference_frac: torch.Tensor, target_frac: torch.Tensor) -> tuple[torch.Tensor, float, list[float]]:
    ref = wrap01(reference_frac)
    tgt = wrap01(target_frac)
    candidate_taus = []
    for idx in range(int(min(ref.shape[0], tgt.shape[0]))):
        tau = wrap01(tgt[idx] - ref[idx])
        candidate_taus.append(tau)
    candidate_taus.append(torch.zeros(3, device=ref.device, dtype=ref.dtype))
    best_tau = candidate_taus[0]
    best_rmse = float('inf')
    scored = []
    for tau in candidate_taus:
        aligned = wrap01(ref + tau.unsqueeze(0))
        rmse = float(torus_rmse(aligned, tgt).detach().item())
        scored.append(float(rmse))
        if rmse < best_rmse:
            best_rmse = rmse
            best_tau = tau.detach().clone()
    return best_tau, best_rmse, scored


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(
        f=case.gt_frac,
        l=case.gt_lattice,
        a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )



def _orbit_template_slices(frame: FixedWyckoffFrame) -> list[tuple[int, int]]:
    slices = []
    cursor = 0
    for orbit in frame.constraint.orbits:
        n = int(orbit.target_indices.numel())
        slices.append((cursor, cursor + n))
        cursor += n
    return slices


def _species_aware_ordered_rmse_torch(source_frac: torch.Tensor, source_z: torch.Tensor, target_frac: torch.Tensor, target_z: torch.Tensor) -> float | None:
    rmse, _status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(source_frac.detach().cpu(), dtype=float),
        source_atomic_numbers=np.asarray(source_z.detach().cpu(), dtype=int),
        target_frac_coords=np.asarray(target_frac.detach().cpu(), dtype=float),
        target_atomic_numbers=np.asarray(target_z.detach().cpu(), dtype=int),
    )
    return None if rmse is None else float(rmse)


def _orbit_hypothesis_audit(case: GraphCase, fixture: dict[str, Any]) -> dict[str, Any]:
    frame = fixture['frame']
    template = fixture['template']
    free_vars = fixture['free_vars']
    expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars, wrap=True)
    template_frac = wrap01(expansion.frac_coords)
    slices = _orbit_template_slices(frame)
    per_orbit = []
    for orbit, (start, stop) in zip(frame.constraint.orbits, slices):
        idx = orbit.target_indices.to(device=case.gt_frac.device, dtype=torch.long)
        gt_block = wrap01(case.gt_frac[idx])
        current_block = wrap01(frame.reference_target_order[idx])
        template_block = wrap01(template_frac[start:stop])
        current_rmse = float(torus_rmse(current_block, gt_block).detach().item())
        # best within-orbit permutation from template block to GT block
        src = template_block.detach().cpu().numpy()
        dst = gt_block.detach().cpu().numpy()
        delta = src[:, None, :] - dst[None, :, :]
        delta = delta - np.round(delta)
        cost = np.sum(delta * delta, axis=-1)
        rows, cols = _hungarian_local(cost)
        permuted_block = template_block[torch.as_tensor(rows, device=template_block.device, dtype=torch.long)]
        dst_order = torch.as_tensor(cols, device=gt_block.device, dtype=torch.long)
        permuted_target = gt_block[dst_order]
        permuted_rmse = float(torus_rmse(permuted_block, permuted_target).detach().item())
        residual_norm = float(torch.linalg.norm((gt_block.reshape(-1) - current_block.reshape(-1) - torch.round(gt_block.reshape(-1) - current_block.reshape(-1)))).detach().item())
        per_orbit.append({
            'label': orbit.label,
            'indices': idx.detach().cpu().tolist(),
            'template_slice': [int(start), int(stop)],
            'current_rmse': current_rmse,
            'best_within_orbit_rmse': permuted_rmse,
            'residual_norm': residual_norm,
            'rank': int(orbit.rank),
            'codim': int(orbit.codim),
            'current_indices': idx.detach().cpu().tolist(),
            'best_perm_cols': cols.tolist(),
        })
    return {
        'per_orbit': per_orbit,
        'max_current_rmse': max((row['current_rmse'] for row in per_orbit), default=0.0),
        'max_best_within_orbit_rmse': max((row['best_within_orbit_rmse'] for row in per_orbit), default=0.0),
    }




def _svd_left_basis_torch(A: torch.Tensor, tol: float = 1.0e-7) -> tuple[torch.Tensor, int]:
    if A.numel() == 0 or A.shape[1] == 0:
        return A.new_zeros((A.shape[0], 0)), 0
    U, S, _ = torch.linalg.svd(A, full_matrices=False)
    threshold = tol * S.max().clamp_min(1.0)
    rank = int((S > threshold).sum().item())
    return U[:, :rank].contiguous(), rank


def _build_transport_corrected_frame(case: GraphCase, fixture: dict[str, Any], *, eps: float = 1.0e-4) -> tuple[FixedWyckoffFrame, list[dict[str, Any]]]:
    base_frame = fixture['frame']
    template = fixture['template']
    bridge = fixture['bridge']
    reference_target_order = base_frame.reference_target_order.detach().clone()
    transport_debug = []
    orbits = []
    coord_cursor = 0
    for site_index, (site, base_orbit) in enumerate(zip(template.site_templates, base_frame.constraint.orbits)):
        mult = int(site.multiplicity)
        start = coord_cursor
        stop = coord_cursor + mult
        target_idx = base_orbit.target_indices.detach().clone().to(device=case.gt_frac.device, dtype=torch.long)
        ref_block = reference_target_order[target_idx].detach().clone()
        A_std = []
        basis = np.asarray(site.anchor_basis, dtype=float)
        rotations = np.asarray(site.rotation_matrices, dtype=float)
        if int(site.dof) > 0:
            for rot in rotations:
                A_std.append(rot @ basis)
            A_std = np.concatenate(A_std, axis=0)
            A_vanilla = transport_standardized_tangent_block_to_vanilla_frame(
                tangent_block=A_std,
                bridge=bridge,
            )
            J = torch.as_tensor(A_vanilla, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
            U, rank = _svd_left_basis_torch(J)
            column_norms = [float(torch.linalg.norm(J[:, dim]).detach().item()) for dim in range(J.shape[1])]
        else:
            J = ref_block.new_zeros((3 * mult, 0))
            U, rank = ref_block.new_zeros((3 * mult, 0)), 0
            column_norms = []
        codim = max(3 * mult - int(rank), 0)
        transport_debug.append({
            'site_index': int(site_index),
            'label': str(site.label),
            'dof': int(site.dof),
            'target_indices': target_idx.detach().cpu().tolist(),
            'column_norms': column_norms,
            'jacobian_shape': list(J.shape),
            'rank': int(rank),
            'codim': int(codim),
            'transport_method': getattr(bridge, 'standardized_to_vanilla_method', None),
        })
        orbits.append(
            SVDOrbitConstraint(
                site_index=int(site_index),
                label=str(site.label),
                target_indices=target_idx,
                reference_template_block=ref_block.detach().clone(),
                U_tangent=U.detach().clone(),
                rank=int(rank),
                codim=int(codim),
            )
        )
        coord_cursor = stop
    constraint = SVDWyckoffConstraint(
        orbits=tuple(orbits),
        codim=int(sum(int(orbit.codim) for orbit in orbits)),
        reference_target_frac=reference_target_order.detach().clone(),
    )
    corrected = FixedWyckoffFrame(
        template=base_frame.template,
        free_vars=base_frame.free_vars,
        assignment=base_frame.assignment,
        tau=base_frame.tau,
        reference_template_order=base_frame.reference_template_order,
        reference_target_order=base_frame.reference_target_order,
        atomic_numbers_template_order=base_frame.atomic_numbers_template_order,
        atomic_numbers_target_order=base_frame.atomic_numbers_target_order,
        constraint=constraint,
        reason=base_frame.reason + '_transport_corrected',
        debug_info={**dict(base_frame.debug_info), 'transport_debug': transport_debug},
    )
    return corrected, transport_debug


def _orbit_branch_gauge_audit(case: GraphCase, fixture: dict[str, Any]) -> dict[str, Any]:
    bridge = fixture['bridge']
    gt_result = fixture['gt_result']
    payload = fixture['symmetry_payload']
    template = fixture['template']
    free_vars = fixture['free_vars']
    frame = fixture['frame']

    standardized_gt_frac = torch.as_tensor(np.asarray(payload.expanded_frac_coords, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    standardized_gt_z = torch.as_tensor(np.asarray(payload.expanded_atomic_numbers, dtype=int), device=case.gt_frac.device, dtype=torch.long)
    expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars, wrap=True)
    standardized_template_frac = wrap01(expansion.frac_coords)
    standardized_template_z = expansion.atomic_numbers.to(device=case.gt_frac.device, dtype=torch.long)
    std_rmse = _species_aware_ordered_rmse_torch(standardized_template_frac, standardized_template_z, standardized_gt_frac, standardized_gt_z)

    standardized_structure = build_structure_from_sample(
        f=standardized_template_frac,
        l=torch.as_tensor(np.asarray(bridge.standardized_structure.lattice.matrix, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1),
        a=standardized_template_z,
        lattice_transform=runner.lattice_transform,
    )
    try:
        remapped_vanilla = transport_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_structure,
            bridge=bridge,
        )
    except Exception:
        remapped_vanilla = map_standardized_structure_to_vanilla_frame(
            standardized_structure=standardized_structure,
            vanilla_reference_structure=build_case_structure(case),
            symprec=1e-2,
            angle_tolerance=5.0,
            stol=0.5,
            ltol=0.3,
        )
    remapped_frac = torch.as_tensor(np.asarray(remapped_vanilla.frac_coords, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    remapped_z = torch.as_tensor(np.asarray(remapped_vanilla.atomic_numbers, dtype=int), device=case.gt_frac.device, dtype=torch.long)
    remapped_ordered_frac, remapped_ordered_z, remapped_assignment, remapped_ordered_rmse = species_align_to_target_order(
        source_frac=remapped_frac,
        source_atomic_numbers=remapped_z,
        target_frac=case.gt_frac,
        target_atomic_numbers=case.atomic_numbers,
    )

    per_site = []
    coord_cursor = 0
    free_cursor = 0
    for site_idx, site in enumerate(template.site_templates):
        mult = int(site.multiplicity)
        start = coord_cursor
        stop = coord_cursor + mult
        gt_anchor = torch.as_tensor(np.asarray(payload.anchor_frac_coords[site_idx], dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        basis = torch.as_tensor(np.asarray(site.anchor_basis, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        offset = torch.as_tensor(np.asarray(site.anchor_offset, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype)
        if int(site.dof) == 0:
            recovered_anchor = offset.clone()
            free_slice_vals = []
        else:
            site_free = free_vars[free_cursor:free_cursor + int(site.dof)]
            recovered_anchor = site_free @ basis.transpose(0, 1) + offset
            free_slice_vals = site_free.detach().cpu().tolist()
            free_cursor += int(site.dof)
        standardized_block = standardized_template_frac[start:stop]
        standardized_gt_block = standardized_gt_frac[start:stop]
        standardized_block_rmse = float(torus_rmse(standardized_block, standardized_gt_block).detach().item())
        target_idx = frame.constraint.orbits[site_idx].target_indices.to(device=case.gt_frac.device, dtype=torch.long)
        vanilla_block = frame.reference_target_order[target_idx]
        gt_block = case.gt_frac[target_idx]
        vanilla_block_rmse = float(torus_rmse(vanilla_block, gt_block).detach().item())
        remapped_block = remapped_ordered_frac[target_idx]
        remapped_block_rmse = float(torus_rmse(remapped_block, gt_block).detach().item())
        per_site.append({
            'site_index': int(site_idx),
            'label': str(site.label),
            'multiplicity': mult,
            'dof': int(site.dof),
            'gt_anchor': gt_anchor.detach().cpu().tolist(),
            'recovered_anchor': recovered_anchor.detach().cpu().tolist(),
            'anchor_rmse': float(torus_rmse(recovered_anchor.unsqueeze(0), gt_anchor.unsqueeze(0)).detach().item()),
            'free_slice': free_slice_vals,
            'standardized_block_rmse': standardized_block_rmse,
            'vanilla_block_rmse': vanilla_block_rmse,
            'remapped_block_rmse': remapped_block_rmse,
            'frame_target_indices': target_idx.detach().cpu().tolist(),
        })
        coord_cursor = stop

    return {
        'standardized_template_to_gt_rmse': std_rmse,
        'remapped_ordered_rmse': remapped_ordered_rmse,
        'remapped_assignment': remapped_assignment.detach().cpu().tolist(),
        'per_site': per_site,
    }


def build_correct_fixture(case: GraphCase) -> dict[str, Any]:
    if case.graph_id in correct_fixtures:
        return correct_fixtures[case.graph_id]
    vanilla_structure = build_case_structure(case)
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=vanilla_structure,
        standardization='conventional',
        symprec=1e-2,
        angle_tolerance=5.0,
        stol=0.5,
        ltol=0.3,
    )
    gt_result = build_pyxtal_wyckoff_result(bridge.standardized_structure, symprec=1e-2, pyxtal_tol=1e-2)
    symmetry_payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=bridge.standardized_structure,
        requested_spacegroup=case.requested_sg,
        tol=1e-2,
    )
    template, free_vars, templates = build_template_and_free_vars_from_diffcsppp_payload(
        payload=symmetry_payload,
        atomic_numbers_standardized=torch.as_tensor(np.asarray(bridge.standardized_atomic_numbers, dtype=int), dtype=torch.long),
        max_templates=MAX_TEMPLATES,
    )
    expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars, wrap=True)
    oracle_rmse, oracle_status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(expansion.frac_coords.detach().cpu(), dtype=float),
        source_atomic_numbers=np.asarray(expansion.atomic_numbers.detach().cpu(), dtype=int),
        target_frac_coords=np.asarray(symmetry_payload.expanded_frac_coords, dtype=float),
        target_atomic_numbers=np.asarray(symmetry_payload.expanded_atomic_numbers, dtype=int),
    )
    bridge_frac, bridge_atomic_numbers, bridge_rmse, bridge_status, ordered_transport = _bridge_standardized_to_vanilla_tensors(case, bridge, symmetry_payload)
    mapped_frac, mapped_atomic_numbers, mapped_rmse, mapped_status = _standardized_template_to_vanilla_tensors(
        case,
        bridge,
        template,
        free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        reference_transport=ordered_transport,
    )
    use_bridge_vanilla = bool(
        oracle_rmse is not None
        and math.isfinite(float(oracle_rmse))
        and float(oracle_rmse) < 1.0e-5
        and bridge_rmse is not None
        and math.isfinite(float(bridge_rmse))
        and (mapped_rmse is None or not math.isfinite(float(mapped_rmse)) or float(bridge_rmse) <= float(mapped_rmse) + 1.0e-8)
    )
    chosen_frac = bridge_frac if use_bridge_vanilla else mapped_frac
    chosen_atomic_numbers = bridge_atomic_numbers if use_bridge_vanilla else mapped_atomic_numbers
    chosen_rmse = bridge_rmse if use_bridge_vanilla else mapped_rmse
    chosen_status = bridge_status if use_bridge_vanilla else mapped_status
    ordered_frac, ordered_atomic_numbers, mapped_to_case_assignment, ordered_rmse = species_align_to_target_order(
        source_frac=chosen_frac,
        source_atomic_numbers=chosen_atomic_numbers,
        target_frac=case.gt_frac,
        target_atomic_numbers=case.atomic_numbers,
    )
    base_frame = build_fixed_wyckoff_frame(
        template=template,
        free_vars=free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        reference_frac_target_order=ordered_frac,
        atomic_numbers_target_order=ordered_atomic_numbers,
        reason='correctW_mapped_vanilla_case_order',
        debug=DEBUG_PRINTS,
    )
    repaired_frame, repair_debug = _build_repaired_frame(case, template, free_vars, base_frame)
    transport_frame, transport_debug = _build_transport_corrected_frame(case, fixture={'frame': base_frame, 'template': template, 'free_vars': free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype), 'bridge': bridge})
    base_c_gt = float(base_frame.constraint.value(case.gt_frac).detach().item())
    repaired_c_gt = float(repaired_frame.constraint.value(case.gt_frac).detach().item())
    transport_c_gt = float(transport_frame.constraint.value(case.gt_frac).detach().item())
    base_hard_rmse = float(torus_rmse(base_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    repaired_hard_rmse = float(torus_rmse(repaired_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    transport_hard_rmse = float(torus_rmse(transport_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    use_repaired_frame = bool(repaired_c_gt + 1.0e-8 < base_c_gt or repaired_hard_rmse + 1.0e-8 < base_hard_rmse)
    use_transport_frame = bool(transport_c_gt + 1.0e-8 < min(base_c_gt, repaired_c_gt) or transport_hard_rmse + 1.0e-8 < min(base_hard_rmse, repaired_hard_rmse))
    frame = transport_frame if use_transport_frame else (repaired_frame if use_repaired_frame else base_frame)
    fixture = {
        'bridge': bridge,
        'transport_metadata': _semantic_transport_metadata(bridge),
        'ordered_reference_transport': ordered_transport,
        'gt_result': gt_result,
        'symmetry_payload': symmetry_payload,
        'template': template,
        'free_vars': free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        'templates': templates,
        'frame': frame,
        'base_frame': base_frame,
        'repaired_frame': repaired_frame,
        'transport_frame': transport_frame,
        'oracle_standardized_rmse': oracle_rmse,
        'oracle_standardized_status': oracle_status,
        'mapped_vanilla_rmse': mapped_rmse,
        'mapped_vanilla_status': mapped_status,
        'bridge_vanilla_rmse': bridge_rmse,
        'bridge_vanilla_status': bridge_status,
        'use_bridge_vanilla': use_bridge_vanilla,
        'mapped_reference_frac': mapped_frac,
        'mapped_reference_atomic_numbers': mapped_atomic_numbers,
        'bridge_reference_frac': bridge_frac,
        'bridge_reference_atomic_numbers': bridge_atomic_numbers,
        'ordered_reference_frac': ordered_frac,
        'ordered_reference_atomic_numbers': ordered_atomic_numbers,
        'mapped_to_case_assignment': mapped_to_case_assignment,
        'ordered_vanilla_rmse': ordered_rmse,
        'base_constraint_gt': base_c_gt,
        'repaired_constraint_gt': repaired_c_gt,
        'base_hard_project_gt_rmse': base_hard_rmse,
        'repaired_hard_project_gt_rmse': repaired_hard_rmse,
        'use_repaired_frame': use_repaired_frame,
        'repair_debug': repair_debug,
        'transport_constraint_gt': transport_c_gt,
        'transport_hard_project_gt_rmse': transport_hard_rmse,
        'use_transport_frame': use_transport_frame,
        'transport_debug': transport_debug,
    }
    correct_fixtures[case.graph_id] = fixture
    print(f"[fixture correct] graph={case.graph_id} sg={case.requested_sg} standardized_sg={bridge.standardized_space_group} template_sites={template.total_sites} free_dims={template.total_free_dims} oracle_rmse={oracle_rmse} mapped_rmse={mapped_rmse} bridge_rmse={bridge_rmse} use_bridge_vanilla={use_bridge_vanilla} ordered_rmse={ordered_rmse} oracle_status={oracle_status} mapped_status={mapped_status} bridge_status={bridge_status} atom_order_match={bool(torch.equal(ordered_atomic_numbers, case.atomic_numbers))}", flush=True)
    print(f"[fixture correct repair] graph={case.graph_id} base_c_gt={base_c_gt:.6f} repaired_c_gt={repaired_c_gt:.6f} transport_c_gt={transport_c_gt:.6f} base_hard_rmse={base_hard_rmse:.6f} repaired_hard_rmse={repaired_hard_rmse:.6f} transport_hard_rmse={transport_hard_rmse:.6f} use_repaired_frame={use_repaired_frame} use_transport_frame={use_transport_frame} repair_debug={repair_debug} transport_debug={transport_debug}", flush=True)
    print(f"[fixture correct order] graph={case.graph_id} mapped_to_case_assignment={mapped_to_case_assignment.detach().cpu().tolist()} frame_assignment={frame.assignment.detach().cpu().tolist()} tau={frame.tau.detach().cpu().tolist()}", flush=True)
    return fixture


def build_wrong_sg_fixture(case: GraphCase) -> dict[str, Any]:
    if case.graph_id in wrong_fixtures:
        return wrong_fixtures[case.graph_id]
    correct = build_correct_fixture(case)
    bridge = correct['bridge']
    for offset in range(1, 231):
        sg = ((int(case.requested_sg) + offset - 1) % 230) + 1
        if sg == int(case.requested_sg):
            continue
        try:
            templates = extract_wyckoff_templates(
                space_group_number=int(sg),
                atomic_numbers=torch.as_tensor(np.asarray(bridge.standardized_atomic_numbers, dtype=int), dtype=torch.long),
                max_templates=12,
                quick=False,
            )
        except Exception:
            continue
        for template in templates[:4]:
            if int(template.total_atoms) != int(case.gt_frac.shape[0]):
                continue
            free_vars = torch.zeros(template.total_free_dims, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
            try:
                mapped_frac, mapped_atomic_numbers, mapped_rmse, mapped_status = _standardized_template_to_vanilla_tensors(
                    case,
                    bridge,
                    template,
                    free_vars,
                    reference_transport=correct.get('ordered_reference_transport'),
                )
                ordered_frac, ordered_atomic_numbers, mapped_to_case_assignment, ordered_rmse = species_align_to_target_order(
                    source_frac=mapped_frac,
                    source_atomic_numbers=mapped_atomic_numbers,
                    target_frac=case.gt_frac,
                    target_atomic_numbers=case.atomic_numbers,
                )
                frame = build_fixed_wyckoff_frame(
                    template=template,
                    free_vars=free_vars,
                    reference_frac_target_order=ordered_frac,
                    atomic_numbers_target_order=ordered_atomic_numbers,
                    reason=f'wrongSG_{sg}_mapped_vanilla_case_order',
                    debug=DEBUG_PRINTS,
                )
                c_wrong = float(frame.constraint.value(case.gt_frac).detach().item())
            except Exception:
                continue
            if int(frame.constraint.codim) <= 0 or not math.isfinite(c_wrong) or c_wrong <= 1.0e-6:
                continue
            fixture = {
                'space_group': int(sg),
                'template': template,
                'free_vars': free_vars,
                'frame': frame,
                'constraint_on_gt': c_wrong,
                'mapped_vanilla_rmse': mapped_rmse,
                'mapped_vanilla_status': mapped_status,
                'ordered_vanilla_rmse': ordered_rmse,
                'mapped_to_case_assignment': mapped_to_case_assignment,
            }
            wrong_fixtures[case.graph_id] = fixture
            print(f"[fixture wrongSG] graph={case.graph_id} picked_sg={sg} codim={frame.constraint.codim} c_wrong_gt={c_wrong:.6f} mapped_rmse={mapped_rmse} ordered_rmse={ordered_rmse} mapped_status={mapped_status}", flush=True)
            print(f"[fixture wrongSG order] graph={case.graph_id} sg={sg} mapped_to_case_assignment={mapped_to_case_assignment.detach().cpu().tolist()} frame_assignment={frame.assignment.detach().cpu().tolist()} tau={frame.tau.detach().cpu().tolist()}", flush=True)
            return fixture
    raise RuntimeError('wrong_space_group_fixture_unavailable')


def build_fixture(case: GraphCase, variant: str) -> dict[str, Any]:
    if str(variant) == 'correctW':
        return build_correct_fixture(case)
    if str(variant) == 'wrongSG':
        return build_wrong_sg_fixture(case)
    raise ValueError(f'Unknown variant={variant!r}')


def algo18_config(*, repeats: int = 1, debug: bool = DEBUG_PRINTS, lambda_noise: float = ALGO18_LAMBDA_NOISE) -> Algorithm18Config:
    return Algorithm18Config(
        repeats=int(repeats),
        proj_steps=int(ALGO18_PROJ_STEPS),
        lr=float(ALGO18_LR),
        lambda_noise=float(lambda_noise),
        optimize_velocity=bool(ALGO18_OPTIMIZE_VELOCITY),
        denoiser_variant=str(ALGO18_DENOISER_VARIANT),
        gradient_clip_norm=10.0,
        mean_free_threshold=1.0e-6,
        lattice_change_threshold=1.0e-8,
        eps=1.0e-8,
        debug=bool(debug),
    )


def algo18_clean_estimate(case: GraphCase, state: KLDMState, *, variant: str = ALGO18_DENOISER_VARIANT) -> torch.Tensor:
    batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
    t_graph, t_nodes = algo18_times(case, float(state.t))
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f_t=state.f,
        v_t=state.v,
        l_t=state.l,
        a_t=state.h,
        t_graph=t_graph,
        t_nodes=t_nodes,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        variant=variant,
    )

print('helpers_ready benchmark_cases=', [case.graph_id for case in benchmark_cases()])


loaded_graphs= [2, 3, 5] sg= [4, 194, 123]
helpers_ready benchmark_cases= [2, 3, 5]


In [23]:
build_correct_fixture_base = build_correct_fixture
print('captured_build_correct_fixture_base=', build_correct_fixture_base.__name__)


captured_build_correct_fixture_base= build_correct_fixture


In [5]:
config_summary = pd.DataFrame([{
    'profile': PROFILE,
    'graphs': ','.join(str(g) for g in GRAPH_IDS),
    'method': ALGORITHM18_MODE,
    'short_name': ALGORITHM18_SHORT_NAME,
    'full_ppr': bool(ALGORITHM18_IS_FULL_PPR),
    't_values': ','.join(f'{t:.2f}' for t in ALGO18_T_VALUES),
    'M_values': ','.join(str(v) for v in ALGO18_M_VALUES),
    'variants': ','.join(ALGO18_VARIANTS),
    'proj_steps': int(ALGO18_PROJ_STEPS),
    'proj_lr': float(ALGO18_LR),
    'lambda_noise': float(ALGO18_LAMBDA_NOISE),
    'optimize_velocity': bool(ALGO18_OPTIMIZE_VELOCITY),
    'denoiser_variant': str(ALGO18_DENOISER_VARIANT),
    'kernel': 'argmin over xi_r,xi_v through D_f -> KLDM renoise from F0*=D_f(F*,V*)',
    'lattice_policy': 'fixed during PPR kernel',
}])
display(config_summary)


,profile,graphs,method,short_name,full_ppr,t_values,M_values,variants,proj_steps,proj_lr,lambda_noise,optimize_velocity,denoiser_variant,kernel,lattice_policy
0,laptop,"2,3,5",kldm_ppr_fractional_velocity_from_scratch,Algorithm18-KLDM-PPR-FromScratch,False,"0.20,0.30","0,1,2","correctW,wrongSG",6,0.02,0.01,True,minus,"argmin over xi_r,xi_v through D_f -> KLDM reno...",fixed during PPR kernel


In [6]:
def frame_problem_row(case: GraphCase) -> dict[str, Any]:
    try:
        correct = build_correct_fixture(case)
        wrong = build_wrong_sg_fixture(case)
        frame = correct['frame']
        base_c_gt = float(correct.get('base_constraint_gt', float('nan')))
        repaired_c_gt = float(correct.get('repaired_constraint_gt', float('nan')))
        base_hard_rmse = float(correct.get('base_hard_project_gt_rmse', float('nan')))
        repaired_hard_rmse = float(correct.get('repaired_hard_project_gt_rmse', float('nan')))
        transport_c_gt = float(correct.get('transport_constraint_gt', float('nan')))
        transport_hard_rmse = float(correct.get('transport_hard_project_gt_rmse', float('nan')))
        c_gt = float(frame.constraint.value(case.gt_frac).detach().item())
        c_ref = float(frame.constraint.value(frame.reference_target_order).detach().item())
        rmse_ref = float(torus_rmse(frame.reference_target_order, case.gt_frac).detach().item())
        rmse_ordered = float(correct.get('ordered_vanilla_rmse', float('nan')))
        z_order_match = bool(torch.equal(correct.get('ordered_reference_atomic_numbers', case.atomic_numbers), case.atomic_numbers))
        best_tau, best_tau_rmse, tau_scan_rmses = _scan_best_global_tau(frame.reference_target_order, case.gt_frac)
        tau_aligned_ref = wrap01(frame.reference_target_order + best_tau.unsqueeze(0))
        hard_proj_gt = frame.hard_project_clean(case.gt_frac)
        rmse_hard = float(torus_rmse(hard_proj_gt, case.gt_frac).detach().item())
        hypothesis = _orbit_hypothesis_audit(case, correct)
        branch_audit = _orbit_branch_gauge_audit(case, correct)
        orbit_debug = hypothesis['per_orbit']
        pass_flag = bool(
            correct['oracle_standardized_rmse'] is not None
            and math.isfinite(float(correct['oracle_standardized_rmse']))
            and float(correct['oracle_standardized_rmse']) < 1.0e-4
            and z_order_match
            and c_gt < 1.0e-4
            and rmse_hard < 5.0e-3
            and c_gt + 1.0e-6 < 0.5 * float(wrong['constraint_on_gt'])
        )
        print(
            f"[frame audit] graph={case.graph_id} oracle_std_rmse={correct['oracle_standardized_rmse']} "
            f"mapped_vanilla_rmse={correct.get('mapped_vanilla_rmse')} bridge_vanilla_rmse={correct.get('bridge_vanilla_rmse')} use_bridge_vanilla={bool(correct.get('use_bridge_vanilla', False))} ordered_vanilla_rmse={rmse_ordered} vanilla_ref_rmse={rmse_ref:.6f} best_tau_rmse={best_tau_rmse:.6f} best_tau={best_tau.detach().cpu().tolist()} base_c_gt={base_c_gt:.6f} repaired_c_gt={repaired_c_gt:.6f} transport_c_gt={transport_c_gt:.6f} base_hard_rmse={base_hard_rmse:.6f} repaired_hard_rmse={repaired_hard_rmse:.6f} transport_hard_rmse={transport_hard_rmse:.6f} use_repaired_frame={bool(correct.get('use_repaired_frame', False))} use_transport_frame={bool(correct.get('use_transport_frame', False))} "
            f"c_gt={c_gt:.6f} c_ref={c_ref:.6f} hard_rmse={rmse_hard:.6f} wrong_c_gt={wrong['constraint_on_gt']:.6f} z_order_match={z_order_match}",
            flush=True,
        )
        print(f"[frame orbit debug] graph={case.graph_id} orbits={orbit_debug}", flush=True)
        print(f"[frame hypothesis] graph={case.graph_id} max_current_orbit_rmse={hypothesis['max_current_rmse']:.6f} max_best_within_orbit_rmse={hypothesis['max_best_within_orbit_rmse']:.6f}", flush=True)
        print(f"[branch/gauge audit] graph={case.graph_id} standardized_template_to_gt_rmse={branch_audit['standardized_template_to_gt_rmse']} remapped_ordered_rmse={branch_audit['remapped_ordered_rmse']} remapped_assignment={branch_audit['remapped_assignment']}", flush=True)
        print(f"[branch/gauge per-site] graph={case.graph_id} per_site={branch_audit['per_site']}", flush=True)
        return {
            'test': 'algo18_frame_problem_audit',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'oracle_standardized_rmse': correct['oracle_standardized_rmse'],
            'oracle_standardized_status': correct['oracle_standardized_status'],
            'mapped_vanilla_rmse': correct.get('mapped_vanilla_rmse'),
            'mapped_vanilla_status': correct.get('mapped_vanilla_status'),
            'bridge_vanilla_rmse': correct.get('bridge_vanilla_rmse'),
            'bridge_vanilla_status': correct.get('bridge_vanilla_status'),
            'use_bridge_vanilla': bool(correct.get('use_bridge_vanilla', False)),
            'ordered_vanilla_rmse': rmse_ordered,
            'best_tau_rmse': best_tau_rmse,
            'base_constraint_gt': base_c_gt,
            'repaired_constraint_gt': repaired_c_gt,
            'base_hard_project_gt_rmse': base_hard_rmse,
            'repaired_hard_project_gt_rmse': repaired_hard_rmse,
            'transport_constraint_gt': transport_c_gt,
            'transport_hard_project_gt_rmse': transport_hard_rmse,
            'use_transport_frame': bool(correct.get('use_transport_frame', False)),
            'transport_debug': str(correct.get('transport_debug')),
            'use_repaired_frame': bool(correct.get('use_repaired_frame', False)),
            'repair_debug': str(correct.get('repair_debug')),
            'best_tau': str(best_tau.detach().cpu().tolist()),
            'tau_scan_rmses': str(tau_scan_rmses),
            'z_order_match': z_order_match,
            'mapped_to_case_assignment': str(correct.get('mapped_to_case_assignment').detach().cpu().tolist() if correct.get('mapped_to_case_assignment') is not None else None),
            'orbit_debug': str(orbit_debug),
            'max_current_orbit_rmse': float(hypothesis['max_current_rmse']),
            'max_best_within_orbit_rmse': float(hypothesis['max_best_within_orbit_rmse']),
            'standardized_template_to_gt_rmse': branch_audit['standardized_template_to_gt_rmse'],
            'remapped_ordered_rmse': branch_audit['remapped_ordered_rmse'],
            'branch_gauge_per_site': str(branch_audit['per_site']),
            'reference_rmse_to_vanilla_gt': rmse_ref,
            'hard_project_gt_rmse': rmse_hard,
            'constraint_gt': c_gt,
            'constraint_ref': c_ref,
            'wrongSG_constraint_on_gt': wrong['constraint_on_gt'],
            'correct_beats_wrong': bool(c_gt < float(wrong['constraint_on_gt'])),
            'frame_valid_for_science': pass_flag,
            'PASS': pass_flag,
            'status': 'PASS' if pass_flag else 'WARN',
        }
    except Exception as exc:
        return error_row('algo18_frame_problem_audit', case.graph_id, exc, 'ALGO18_FRAME_AUDIT_ERROR')

rows = [frame_problem_row(case) for case in benchmark_cases()]
display(dataframe_result('algo18_frame_problem_audit', rows))


rows_branch = []
for case in benchmark_cases():
    try:
        correct = build_correct_fixture(case)
        audit = _orbit_branch_gauge_audit(case, correct)
        rows_branch.append({
            'test': 'algo18_branch_gauge_deep_audit',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'standardized_template_to_gt_rmse': audit['standardized_template_to_gt_rmse'],
            'remapped_ordered_rmse': audit['remapped_ordered_rmse'],
            'remapped_assignment': str(audit['remapped_assignment']),
            'per_site': str(audit['per_site']),
            'PASS': True,
            'status': 'INFO',
        })
    except Exception as exc:
        rows_branch.append(error_row('algo18_branch_gauge_deep_audit', case.graph_id, exc, 'ALGO18_BRANCH_GAUGE_AUDIT_ERROR'))
display(dataframe_result('algo18_branch_gauge_deep_audit', rows_branch))


[algo18 frame] reason=correctW_mapped_vanilla_case_order rmse_ref=0.150392 tau=[-0.0001742132008075714, 0.013845639303326607, -0.0005153026431798935] n_atoms=16 n_free=24
[algo18 frame] reason=correctW_mapped_vanilla_case_order_orbit_repaired rmse_ref=0.182105 tau=[-0.00017422158271074295, 0.013845634646713734, -0.0005152896046638489] n_atoms=16 n_free=24
[fixture correct] graph=2 sg=4 standardized_sg=4 template_sites=8 free_dims=24 oracle_rmse=1.3645279471209316e-08 mapped_rmse=0.12686224471964377 bridge_rmse=0.12686224566304768 use_bridge_vanilla=True ordered_rmse=0.12686224281787872 oracle_status=ok mapped_status=ordered_semantic_transport bridge_status=ordered_semantic_transport atom_order_match=True
[fixture correct repair] graph=2 base_c_gt=0.015204 repaired_c_gt=0.062371 transport_c_gt=0.026556 base_hard_rmse=0.087190 repaired_hard_rmse=0.176595 transport_hard_rmse=0.115231 use_repaired_frame=False use_transport_frame=False repair_debug=[{'label': '2a', 'size': 2, 'changed': Fal

/tmp/ipykernel_7475/2706896954.py:594: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  l=torch.as_tensor(np.asarray(bridge.standardized_structure.lattice.matrix, dtype=float), device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1),


[fixture correct] graph=3 sg=194 standardized_sg=194 template_sites=2 free_dims=1 oracle_rmse=1.3753149023124934e-08 mapped_rmse=0.16896528764204177 bridge_rmse=0.16896528764204177 use_bridge_vanilla=True ordered_rmse=0.16896529495716095 oracle_status=ok mapped_status=ordered_semantic_transport bridge_status=ordered_semantic_transport atom_order_match=True
[fixture correct repair] graph=3 base_c_gt=0.077377 repaired_c_gt=0.081081 transport_c_gt=0.075478 base_hard_rmse=0.272311 repaired_hard_rmse=0.274084 transport_hard_rmse=0.254162 use_repaired_frame=False use_transport_frame=True repair_debug=[{'label': '2d', 'size': 2, 'changed': False, 'old_indices': [0, 1], 'new_indices': [0, 1], 'rmse_before': 0.4345264434814453, 'rmse_after': 0.25087396526012407}, {'label': '6h', 'size': 6, 'changed': True, 'old_indices': [5, 7, 6, 2, 4, 3], 'new_indices': [7, 2, 6, 4, 5, 3], 'rmse_before': 0.4892207980155945, 'rmse_after': 0.17640841886788106}] transport_debug=[{'site_index': 0, 'label': '2d', 

,test,graph,space_group,oracle_standardized_rmse,oracle_standardized_status,mapped_vanilla_rmse,mapped_vanilla_status,bridge_vanilla_rmse,bridge_vanilla_status,use_bridge_vanilla,...,branch_gauge_per_site,reference_rmse_to_vanilla_gt,hard_project_gt_rmse,constraint_gt,constraint_ref,wrongSG_constraint_on_gt,correct_beats_wrong,frame_valid_for_science,PASS,status
0,algo18_frame_problem_audit,2,4,1.364528e-08,ok,1.268622e-01,ordered_semantic_transport,1.268622e-01,ordered_semantic_transport,True,...,"[{'site_index': 0, 'label': '2a', 'multiplicit...",1.470720e-01,8.718965e-02,1.520407e-02,0.0,0.061650,True,False,False,WARN
1,algo18_frame_problem_audit,3,194,1.375315e-08,ok,1.689653e-01,ordered_semantic_transport,1.689653e-01,ordered_semantic_transport,True,...,"[{'site_index': 0, 'label': '2d', 'multiplicit...",2.748976e-01,2.541618e-01,7.547808e-02,0.0,0.056417,False,False,False,WARN
2,algo18_frame_problem_audit,5,123,9.197196e-09,ok,1.713111e-17,ordered_semantic_transport,1.713111e-17,ordered_semantic_transport,True,...,"[{'site_index': 0, 'label': '1a', 'multiplicit...",4.894604e-18,4.894604e-18,2.515501e-35,0.0,0.107143,True,True,True,PASS


,test,graph,space_group,standardized_template_to_gt_rmse,remapped_ordered_rmse,remapped_assignment,per_site,PASS,status
0,algo18_branch_gauge_deep_audit,2,4,8.867965e-09,1.268622e-01,"[3, 1, 2, 0, 4, 5, 9, 7, 6, 8, 11, 10, 15, 14,...","[{'site_index': 0, 'label': '2a', 'multiplicit...",True,INFO
1,algo18_branch_gauge_deep_audit,3,194,0.000000e+00,1.689653e-01,"[0, 1, 2, 6, 7, 5, 3, 4]","[{'site_index': 0, 'label': '2d', 'multiplicit...",True,INFO
2,algo18_branch_gauge_deep_audit,5,123,0.000000e+00,5.139334e-17,"[0, 1, 3, 2, 4, 5, 6]","[{'site_index': 0, 'label': '1a', 'multiplicit...",True,INFO


In [24]:
from dataclasses import dataclass

from kldmPlus.symmetry import diffcsppp_backend as diffcsppp_backend


@dataclass(frozen=True)
class DiffCSPPPOperatorConstraint:
    payload: Any
    payload_to_case_assignment: torch.Tensor
    source_atomic_numbers: torch.Tensor

    def project(self, z_clean: torch.Tensor) -> torch.Tensor:
        target_frac = torch.as_tensor(np.asarray(self.payload.expanded_frac_coords, dtype=float), device=z_clean.device, dtype=z_clean.dtype)
        target_atomic_numbers = torch.as_tensor(np.asarray(self.payload.expanded_atomic_numbers, dtype=int), device=z_clean.device, dtype=torch.long)
        payload_to_case = self.payload_to_case_assignment.to(device=z_clean.device, dtype=torch.long).reshape(-1)
        case_to_payload = torch.empty_like(payload_to_case)
        case_to_payload[payload_to_case] = torch.arange(payload_to_case.numel(), device=z_clean.device, dtype=torch.long)
        aligned_frac = wrap01(z_clean)[case_to_payload]
        projected = diffcsppp_backend.project_expanded_frac_to_anchor_space(
            self.payload,
            np.asarray(aligned_frac.detach().cpu(), dtype=float),
        )
        projected_payload = torch.as_tensor(np.asarray(projected['lifted_expanded_frac_coords'], dtype=float), device=z_clean.device, dtype=z_clean.dtype)
        projected_case = projected_payload[payload_to_case]
        return wrap01(projected_case)

    def value(self, z_clean: torch.Tensor) -> torch.Tensor:
        projected = self.project(z_clean)
        return torus_rmse(wrap01(z_clean), projected).square()


@dataclass(frozen=True)
class DiffCSPPPOperatorFrame:
    payload: Any
    constraint: DiffCSPPPOperatorConstraint
    reference_target_order: torch.Tensor
    atomic_numbers_target_order: torch.Tensor
    reason: str
    debug_info: dict[str, Any]

    def hard_project_clean(self, z_clean: torch.Tensor) -> torch.Tensor:
        return self.constraint.project(z_clean)


build_correct_fixture_base = globals().get('build_correct_fixture_base', build_correct_fixture)


def build_diffcsppp_operator_frame(case: GraphCase, fixture: dict[str, Any]) -> DiffCSPPPOperatorFrame:
    payload = fixture['symmetry_payload']
    assignment = fixture['mapped_to_case_assignment'].detach().clone()
    constraint = DiffCSPPPOperatorConstraint(
        payload=payload,
        payload_to_case_assignment=assignment,
        source_atomic_numbers=case.atomic_numbers.detach().clone(),
    )
    projected_gt = constraint.project(case.gt_frac)
    projected_rmse = float(torus_rmse(projected_gt, case.gt_frac).detach().item())
    return DiffCSPPPOperatorFrame(
        payload=payload,
        constraint=constraint,
        reference_target_order=fixture['ordered_reference_frac'].detach().clone(),
        atomic_numbers_target_order=fixture['ordered_reference_atomic_numbers'].detach().clone(),
        reason='oracle_diffcsppp_operator_frame',
        debug_info={
            'gt_projection_rmse': projected_rmse,
            'payload_spacegroup': int(payload.spacegroup),
            'payload_signature': str(payload.site_signature),
            'payload_to_case_assignment': assignment.detach().cpu().tolist(),
        },
    )


def build_correct_fixture(case: GraphCase) -> dict[str, Any]:
    fixture = build_correct_fixture_base(case)
    if 'operator_frame' not in fixture:
        fixture = dict(fixture)
        fixture['svd_frame'] = fixture.get('frame')
        fixture['operator_frame'] = build_diffcsppp_operator_frame(case, fixture)
        fixture['operator_frame_rmse'] = fixture['operator_frame'].debug_info['gt_projection_rmse']
    return fixture


In [25]:
from kldmPlus.symmetry import diffcsppp_backend as diffcsppp_backend


def diffcsppp_contract_and_anchor_candidate(case: GraphCase) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    correct = build_correct_fixture(case)
    payload = correct['symmetry_payload']
    bridge = correct['bridge']
    base_frame = correct['base_frame']
    current_frame = correct['frame']

    payload_atomic_numbers = torch.as_tensor(np.asarray(payload.expanded_atomic_numbers, dtype=int), dtype=torch.long)
    payload_lift = diffcsppp_backend.reconstruct_expanded_frac_from_anchor_coords(
        payload,
        payload.anchor_frac_coords,
    )
    payload_lift_rmse, payload_lift_status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(payload_lift, dtype=float),
        source_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        target_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        target_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
    )
    payload_projection = diffcsppp_backend.project_expanded_frac_to_anchor_space(
        payload,
        payload.expanded_frac_coords,
    )

    audit_template, audit_free_vars, audit_templates = build_template_and_free_vars_from_diffcsppp_payload(
        payload=payload,
        atomic_numbers_standardized=payload_atomic_numbers,
        max_templates=MAX_TEMPLATES,
    )
    audit_expansion = expand_wyckoff_template_torch(template=audit_template, free_vars=audit_free_vars, wrap=True)
    audit_expansion_rmse, audit_expansion_status = species_aware_torus_rmse(
        source_frac_coords=np.asarray(audit_expansion.frac_coords.detach().cpu(), dtype=float),
        source_atomic_numbers=np.asarray(audit_expansion.atomic_numbers.detach().cpu(), dtype=int),
        target_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        target_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
    )

    payload_site_slices = diffcsppp_backend.payload_site_slices(payload)
    anchor_index = np.asarray(payload.anchor_index, dtype=int).reshape(-1)
    anchor_index_nondecreasing = bool(np.all(anchor_index[1:] >= anchor_index[:-1])) if anchor_index.size > 1 else True
    anchor_slice_count = int(len(payload_site_slices))
    anchor_site_count = int(len(payload.wyckoff_letters))

    candidate_ordered_frac, candidate_ordered_atomic_numbers, candidate_assignment, candidate_ordered_rmse = species_align_to_target_order(
        source_frac=audit_expansion.frac_coords,
        source_atomic_numbers=audit_expansion.atomic_numbers,
        target_frac=case.gt_frac,
        target_atomic_numbers=case.atomic_numbers,
    )
    anchor_candidate_frame = build_fixed_wyckoff_frame(
        template=audit_template,
        free_vars=audit_free_vars.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        reference_frac_target_order=candidate_ordered_frac,
        atomic_numbers_target_order=candidate_ordered_atomic_numbers,
        reason='oracle_diffcsppp_anchor_parameterization',
        debug=DEBUG_PRINTS,
    )

    base_c_gt = float(base_frame.constraint.value(case.gt_frac).detach().item())
    current_c_gt = float(current_frame.constraint.value(case.gt_frac).detach().item())
    candidate_c_gt = float(anchor_candidate_frame.constraint.value(case.gt_frac).detach().item())
    base_hard_rmse = float(torus_rmse(base_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    current_hard_rmse = float(torus_rmse(current_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    candidate_hard_rmse = float(torus_rmse(anchor_candidate_frame.hard_project_clean(case.gt_frac), case.gt_frac).detach().item())
    candidate_wins_base = bool(candidate_c_gt + 1.0e-8 < base_c_gt or candidate_hard_rmse + 1.0e-8 < base_hard_rmse)
    candidate_wins_current = bool(candidate_c_gt + 1.0e-8 < current_c_gt or candidate_hard_rmse + 1.0e-8 < current_hard_rmse)

    correct['oracle_anchor_candidate_frame'] = anchor_candidate_frame
    correct['oracle_anchor_candidate_metrics'] = {
        'candidate_c_gt': candidate_c_gt,
        'candidate_hard_rmse': candidate_hard_rmse,
        'candidate_ordered_rmse': float(candidate_ordered_rmse),
        'candidate_wins_base': candidate_wins_base,
        'candidate_wins_current': candidate_wins_current,
    }
    if candidate_wins_current:
        correct['frame'] = anchor_candidate_frame
        correct['frame_selection_reason'] = 'oracle_diffcsppp_anchor_parameterization_selected'
        correct['use_oracle_anchor_candidate_frame'] = True
    else:
        correct['frame_selection_reason'] = getattr(current_frame, 'reason', 'current_frame')
        correct['use_oracle_anchor_candidate_frame'] = False

    audit_rows = [
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'oracle_spacegroup',
            'observed': int(payload.spacegroup),
            'expected': int(case.requested_sg),
            'PASS': int(payload.spacegroup) == int(case.requested_sg),
            'status': 'PASS' if int(payload.spacegroup) == int(case.requested_sg) else 'FAIL',
        },
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'anchor_index_contract',
            'observed': f'{anchor_site_count} sites / {anchor_slice_count} slices / {payload.num_atoms} atoms',
            'expected': f'{anchor_site_count} contiguous site slices',
            'PASS': bool(anchor_index_nondecreasing and anchor_slice_count == anchor_site_count),
            'status': 'PASS' if bool(anchor_index_nondecreasing and anchor_slice_count == anchor_site_count) else 'FAIL',
        },
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'anchor_reconstruction',
            'observed': payload_lift_rmse,
            'expected': '< 1e-5',
            'PASS': bool(payload_lift_rmse is not None and payload_lift_status == 'ok' and float(payload_lift_rmse) < 1.0e-5),
            'status': 'PASS' if bool(payload_lift_rmse is not None and payload_lift_status == 'ok' and float(payload_lift_rmse) < 1.0e-5) else 'FAIL',
        },
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'anchor_projection',
            'observed': float(payload_projection['rmse']),
            'expected': '< 1e-5',
            'PASS': bool(payload_projection['rmse'] is not None and float(payload_projection['rmse']) < 1.0e-5),
            'status': 'PASS' if bool(payload_projection['rmse'] is not None and float(payload_projection['rmse']) < 1.0e-5) else 'FAIL',
        },
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'template_signature',
            'observed': str(flatten_site_signature(audit_template)),
            'expected': str(payload.site_signature),
            'PASS': bool(flatten_site_signature(audit_template) == payload.site_signature),
            'status': 'PASS' if bool(flatten_site_signature(audit_template) == payload.site_signature) else 'FAIL',
        },
        {
            'test': 'algo18_diffcsppp_contract_audit',
            'graph': case.graph_id,
            'step': 'template_reconstruction',
            'observed': audit_expansion_rmse,
            'expected': '< 1e-5',
            'PASS': bool(audit_expansion_status == 'ok' and audit_expansion_rmse is not None and float(audit_expansion_rmse) < 1.0e-5),
            'status': 'PASS' if bool(audit_expansion_status == 'ok' and audit_expansion_rmse is not None and float(audit_expansion_rmse) < 1.0e-5) else 'FAIL',
        },
    ]

    candidate_row = {
        'test': 'algo18_diffcsppp_anchor_frame_candidate',
        'graph': case.graph_id,
        'space_group': int(case.requested_sg),
        'template_signature': str(flatten_site_signature(audit_template)),
        'ordered_rmse_to_case': float(candidate_ordered_rmse),
        'base_constraint_gt': base_c_gt,
        'current_constraint_gt': current_c_gt,
        'candidate_constraint_gt': candidate_c_gt,
        'base_hard_project_gt_rmse': base_hard_rmse,
        'current_hard_project_gt_rmse': current_hard_rmse,
        'candidate_hard_project_gt_rmse': candidate_hard_rmse,
        'candidate_wins_base': candidate_wins_base,
        'candidate_wins_current': candidate_wins_current,
        'anchor_candidate_reason': anchor_candidate_frame.reason,
        'selected_frame_reason': correct['frame_selection_reason'],
        'PASS': candidate_wins_current,
        'status': 'PASS' if candidate_wins_current else 'WARN',
    }

    print(
        f"[diffcsppp audit] graph={case.graph_id} sg={case.requested_sg} payload_sg={payload.spacegroup} "
        f"anchor_recon_rmse={payload_lift_rmse} anchor_proj_rmse={payload_projection['rmse']} template_rmse={audit_expansion_rmse} "
        f"template_signature_match={bool(flatten_site_signature(audit_template) == payload.site_signature)}",
        flush=True,
    )
    print(
        f"[diffcsppp candidate] graph={case.graph_id} base_c={base_c_gt:.6f} current_c={current_c_gt:.6f} candidate_c={candidate_c_gt:.6f} "
        f"base_hard={base_hard_rmse:.6f} current_hard={current_hard_rmse:.6f} candidate_hard={candidate_hard_rmse:.6f} "
        f"wins_base={candidate_wins_base} wins_current={candidate_wins_current} selected={correct['frame_selection_reason']}",
        flush=True,
    )
    return audit_rows, candidate_row


audit_rows = []
anchor_candidate_rows = []
for case in benchmark_cases():
    try:
        case_audit_rows, case_candidate_row = diffcsppp_contract_and_anchor_candidate(case)
        audit_rows.extend(case_audit_rows)
        anchor_candidate_rows.append(case_candidate_row)
    except Exception as exc:
        audit_rows.append(error_row('algo18_diffcsppp_contract_audit', case.graph_id, exc, 'ALGO18_DIFFCSPPP_CONTRACT_AUDIT_ERROR'))
        anchor_candidate_rows.append(error_row('algo18_diffcsppp_anchor_frame_candidate', case.graph_id, exc, 'ALGO18_DIFFCSPPP_ANCHOR_CANDIDATE_ERROR'))


display(dataframe_result('algo18_diffcsppp_contract_audit', audit_rows))
display(dataframe_result('algo18_diffcsppp_anchor_frame_candidate', anchor_candidate_rows))


[algo18 frame] reason=correctW_mapped_vanilla_case_order rmse_ref=0.150392 tau=[-0.0001742132008075714, 0.013845639303326607, -0.0005153026431798935] n_atoms=16 n_free=24
[algo18 frame] reason=correctW_mapped_vanilla_case_order_orbit_repaired rmse_ref=0.182105 tau=[-0.00017422158271074295, 0.013845634646713734, -0.0005152896046638489] n_atoms=16 n_free=24
[fixture correct] graph=2 sg=4 standardized_sg=4 template_sites=8 free_dims=24 oracle_rmse=1.3645279471209316e-08 mapped_rmse=0.12686224471964377 bridge_rmse=0.12686224566304768 use_bridge_vanilla=True ordered_rmse=0.12686224281787872 oracle_status=ok mapped_status=ordered_semantic_transport bridge_status=ordered_semantic_transport atom_order_match=True
[fixture correct repair] graph=2 base_c_gt=0.015204 repaired_c_gt=0.062371 transport_c_gt=0.026556 base_hard_rmse=0.087190 repaired_hard_rmse=0.176595 transport_hard_rmse=0.115231 use_repaired_frame=False use_transport_frame=False repair_debug=[{'label': '2a', 'size': 2, 'changed': Fal

,test,graph,step,observed,expected,PASS,status
0,algo18_diffcsppp_contract_audit,2,oracle_spacegroup,4,4,True,PASS
1,algo18_diffcsppp_contract_audit,2,anchor_index_contract,8 sites / 8 slices / 16 atoms,8 contiguous site slices,True,PASS
2,algo18_diffcsppp_contract_audit,2,anchor_reconstruction,0.0,< 1e-5,True,PASS
3,algo18_diffcsppp_contract_audit,2,anchor_projection,0.0,< 1e-5,True,PASS
4,algo18_diffcsppp_contract_audit,2,template_signature,"((5, '2a'), (5, '2a'), (26, '2a'), (26, '2a'),...","((5, '2a'), (5, '2a'), (26, '2a'), (26, '2a'),...",True,PASS
5,algo18_diffcsppp_contract_audit,2,template_reconstruction,0.0,< 1e-5,True,PASS
6,algo18_diffcsppp_contract_audit,3,oracle_spacegroup,194,194,True,PASS
7,algo18_diffcsppp_contract_audit,3,anchor_index_contract,2 sites / 2 slices / 8 atoms,2 contiguous site slices,True,PASS
8,algo18_diffcsppp_contract_audit,3,anchor_reconstruction,0.0,< 1e-5,True,PASS
9,algo18_diffcsppp_contract_audit,3,anchor_projection,0.0,< 1e-5,True,PASS


,test,graph,space_group,template_signature,ordered_rmse_to_case,base_constraint_gt,current_constraint_gt,candidate_constraint_gt,base_hard_project_gt_rmse,current_hard_project_gt_rmse,candidate_hard_project_gt_rmse,candidate_wins_base,candidate_wins_current,anchor_candidate_reason,selected_frame_reason,PASS,status
0,algo18_diffcsppp_anchor_frame_candidate,2,4,"((5, '2a'), (5, '2a'), (26, '2a'), (26, '2a'),...",0.118994,1.520407e-02,1.520407e-02,3.595294e-16,8.718965e-02,8.718965e-02,1.719119e-08,True,True,oracle_diffcsppp_anchor_parameterization,oracle_diffcsppp_anchor_parameterization_selected,True,PASS
1,algo18_diffcsppp_anchor_frame_candidate,3,194,"((22, '6h'), (72, '2d'))",0.190583,7.737740e-02,7.547808e-02,3.778062e-02,2.723111e-01,2.541618e-01,1.902799e-01,True,True,oracle_diffcsppp_anchor_parameterization,oracle_diffcsppp_anchor_parameterization_selected,True,PASS
2,algo18_diffcsppp_anchor_frame_candidate,5,123,"((27, '1b'), (31, '1c'), (31, '4i'), (69, '1a'))",0.000000,2.515501e-35,2.515501e-35,0.000000e+00,4.894604e-18,4.894604e-18,0.000000e+00,False,False,oracle_diffcsppp_anchor_parameterization,correctW_mapped_vanilla_case_order,False,WARN


In [8]:
def denoiser_tests_rows(case: GraphCase, *, t_value: float) -> list[dict[str, Any]]:
    rows = []
    try:
        state = algo18_state_from_noisy_gt(case, t_value=float(t_value), seed=SAMPLE_SEED)
        batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
        t_graph, t_nodes = algo18_times(case, float(t_value))
        shape_minus, shape_plus = denoiser_shape_and_finite_sanity(
            model=runner.model,
            f_t=state.f,
            v_t=state.v,
            l_t=state.l,
            a_t=state.h,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
        )
        for item in (shape_minus, shape_plus):
            rows.append({
                'test': 'algo18_denoiser_shape_finite',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't_value': float(t_value),
                'variant': item.variant,
                'finite': bool(item.finite),
                'shape_ok': bool(item.shape_ok),
                'min_value': item.min_value,
                'max_value': item.max_value,
                'PASS': bool(item.finite and item.shape_ok),
                'status': 'PASS' if bool(item.finite and item.shape_ok) else 'WARN',
            })
        sign = oracle_clean_denoiser_sign_check(
            model=runner.model,
            f0=case.gt_frac,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
        )
        better = 'minus' if sign.minus_rmse <= sign.plus_rmse else 'plus'
        rows.append({
            'test': 'algo18_oracle_sign_test',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'minus_rmse': float(sign.minus_rmse),
            'plus_rmse': float(sign.plus_rmse),
            'better_variant': better,
            'PASS': bool(min(float(sign.minus_rmse), float(sign.plus_rmse)) < 1.0e-4),
            'status': 'PASS' if bool(min(float(sign.minus_rmse), float(sign.plus_rmse)) < 1.0e-4) else 'WARN',
        })
        for variant in ('minus', 'plus'):
            clean_hat = algo18_clean_estimate(case, state, variant=variant)
            frac_rmse = float(torus_rmse(clean_hat, case.gt_frac).detach().item())
            eval_clean = evaluate_arrays(case, clean_hat, state.l, case.atomic_numbers)
            print(f"[D2] graph={case.graph_id} t={t_value:.2f} variant={variant} frac_rmse={frac_rmse:.6f} valid={eval_clean['valid']} rmse={eval_clean['rmse']}", flush=True)
            rows.append({
                'test': 'algo18_closed_form_denoiser_quality',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't_value': float(t_value),
                'variant': variant,
                'frac_RMSE_to_GT': frac_rmse,
                'match': bool(eval_clean.get('match', False)),
                'valid_structure': bool(eval_clean.get('valid', False)),
                'RMSE': eval_clean.get('rmse'),
                'min_pair_distance': eval_clean.get('min_pair_distance'),
                'metric_source': eval_clean.get('metric_source'),
                'PASS': bool(torch.isfinite(clean_hat).all().item() and math.isfinite(frac_rmse)),
                'status': 'PASS' if bool(torch.isfinite(clean_hat).all().item() and math.isfinite(frac_rmse)) else 'WARN',
            })
        return rows
    except Exception as exc:
        return [error_row('algo18_denoiser_bundle', case.graph_id, exc, 'ALGO18_DENOISER_BUNDLE_ERROR', t_value=t_value)]

rows = []
for case in benchmark_cases():
    for t_value in ALGO18_T_VALUES:
        rows.extend(denoiser_tests_rows(case, t_value=float(t_value)))
display(dataframe_result('algo18_denoiser_tests', rows))


[D2] graph=2 t=0.20 variant=minus frac_rmse=0.004987 valid=True rmse=0.020508346502664716
[D2] graph=2 t=0.20 variant=plus frac_rmse=0.036775 valid=True rmse=0.16348372953777862
[D2] graph=2 t=0.30 variant=minus frac_rmse=0.005125 valid=True rmse=0.021659554361562174
[D2] graph=2 t=0.30 variant=plus frac_rmse=0.058832 valid=True rmse=0.26806182753645796
[D2] graph=3 t=0.20 variant=minus frac_rmse=0.002120 valid=True rmse=0.008249033870010712
[D2] graph=3 t=0.20 variant=plus frac_rmse=0.023954 valid=True rmse=0.0845412833553268
[D2] graph=3 t=0.30 variant=minus frac_rmse=0.002787 valid=True rmse=0.011017211260690479
[D2] graph=3 t=0.30 variant=plus frac_rmse=0.050695 valid=True rmse=0.16727775928335314
[D2] graph=5 t=0.20 variant=minus frac_rmse=0.000650 valid=True rmse=0.0025496044486608164
[D2] graph=5 t=0.20 variant=plus frac_rmse=0.020703 valid=True rmse=0.0690986768273712
[D2] graph=5 t=0.30 variant=minus frac_rmse=0.001218 valid=True rmse=0.004290816358256232
[D2] graph=5 t=0.30 v

,test,graph,space_group,t_value,variant,finite,shape_ok,min_value,max_value,PASS,status,minus_rmse,plus_rmse,better_variant,frac_RMSE_to_GT,match,valid_structure,RMSE,min_pair_distance,metric_source
0,algo18_denoiser_shape_finite,2,4,0.2,minus,True,True,-0.499472,0.499304,True,PASS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,algo18_denoiser_shape_finite,2,4,0.2,plus,True,True,-0.496994,0.498307,True,PASS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,algo18_oracle_sign_test,2,4,0.2,NaN,NaN,NaN,NaN,NaN,True,PASS,8.603189e-09,8.603189e-09,minus,NaN,NaN,NaN,NaN,NaN,NaN
3,algo18_closed_form_denoiser_quality,2,4,0.2,minus,NaN,NaN,NaN,NaN,True,PASS,NaN,NaN,NaN,0.004987,True,True,0.020508,2.029426,sample_evaluation.evaluate_csp_reconstruction
4,algo18_closed_form_denoiser_quality,2,4,0.2,plus,NaN,NaN,NaN,NaN,True,PASS,NaN,NaN,NaN,0.036775,True,True,0.163484,1.446316,sample_evaluation.evaluate_csp_reconstruction
5,algo18_denoiser_shape_finite,2,4,0.3,minus,True,True,-0.499397,0.498571,True,PASS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,algo18_denoiser_shape_finite,2,4,0.3,plus,True,True,-0.463915,0.496739,True,PASS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,algo18_oracle_sign_test,2,4,0.3,NaN,NaN,NaN,NaN,NaN,True,PASS,1.290478e-08,1.290478e-08,minus,NaN,NaN,NaN,NaN,NaN,NaN
8,algo18_closed_form_denoiser_quality,2,4,0.3,minus,NaN,NaN,NaN,NaN,True,PASS,NaN,NaN,NaN,0.005125,True,True,0.021660,2.016674,sample_evaluation.evaluate_csp_reconstruction
9,algo18_closed_form_denoiser_quality,2,4,0.3,plus,NaN,NaN,NaN,NaN,True,PASS,NaN,NaN,NaN,0.058832,True,True,0.268062,0.957917,sample_evaluation.evaluate_csp_reconstruction


In [9]:
def wyckoff_kinematic_row(case: GraphCase, *, t_value: float) -> dict[str, Any]:
    try:
        fixture = build_correct_fixture(case)
        frame = fixture['frame']
        state = algo18_state_from_noisy_gt(case, t_value=float(t_value), seed=SAMPLE_SEED)
        batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
        t_graph, t_nodes = algo18_times(case, float(t_value))

        orth_err = 0.0
        tangent_vals = []
        boundary_vals = []
        for orbit in frame.constraint.orbits:
            U = orbit.U_tangent
            if U.numel() != 0:
                orth_err = max(orth_err, float(torch.linalg.norm(U.transpose(0, 1) @ U - torch.eye(U.shape[1], device=U.device, dtype=U.dtype)).detach().item()))
        c_ref = float(frame.constraint.value(frame.reference_target_order).detach().item())
        for scale in (0.01, 0.05):
            free_vars = frame.free_vars + scale * torch.randn_like(frame.free_vars)
            tangent_vals.append(float(frame.constraint.value(frame.expand_target_order(free_vars)).detach().item()))
        boundary_point = frame.reference_target_order.detach().clone()
        boundary_point[:1] = wrap01(boundary_point[:1] + torch.tensor([[0.49, -0.49, 0.49]], device=boundary_point.device, dtype=boundary_point.dtype))
        boundary_c = float(frame.constraint.value(boundary_point).detach().item())
        boundary_vals.append(boundary_c)

        grad = ppr_objective_gradient_sanity(
            model=runner.model,
            frame=frame,
            state=algo18_to_backend_state(state),
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            config=algo18_config(repeats=1, debug=False),
        )

        centered_v, mean_free_norm = graph_center_velocity(state.v, batch_view.batch)
        renoise_only = kldm_renoise_from_f0(
            model=runner.model,
            f0_star=algo18_clean_estimate(case, state),
            l_t=state.l,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            debug=DEBUG_PRINTS,
            debug_prefix=f'[K2 graph={case.graph_id} t={t_value:.2f}]',
        )
        print(f"[W/K] graph={case.graph_id} t={t_value:.2f} c_ref={c_ref:.3e} orth_err={orth_err:.3e} tangent_vals={tangent_vals} boundary_c={boundary_c:.6f} grad_r={grad['grad_xi_r_norm']:.3e} grad_v={grad['grad_xi_v_norm']:.3e}", flush=True)
        pass_flag = bool(
            c_ref < 1.0e-5
            and orth_err < 1.0e-5
            and grad['finite_grad']
            and renoise_only.finite_ok
            and renoise_only.mean_free_norm < 1.0e-6
            and renoise_only.lattice_changed_norm < 1.0e-8
        )
        return {
            'test': 'algo18_wyckoff_and_kinematic_sanity',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'constraint_ref': c_ref,
            'orthonormality_error_max': orth_err,
            'tangent_constraint_mean': float(np.mean(tangent_vals)) if tangent_vals else 0.0,
            'boundary_constraint': boundary_c,
            'grad_xi_r_norm': grad['grad_xi_r_norm'],
            'grad_xi_v_norm': grad['grad_xi_v_norm'],
            'finite_grad': bool(grad['finite_grad']),
            'state_mean_free_norm': float(mean_free_norm),
            'renoise_finite_ok': bool(renoise_only.finite_ok),
            'renoise_mean_free_norm': float(renoise_only.mean_free_norm),
            'renoise_lattice_changed_norm': float(renoise_only.lattice_changed_norm),
            'PASS': pass_flag,
            'status': 'PASS' if pass_flag else 'WARN',
        }
    except Exception as exc:
        return error_row('algo18_wyckoff_and_kinematic_sanity', case.graph_id, exc, 'ALGO18_WK_SANITY_ERROR', t_value=t_value)

rows = [wyckoff_kinematic_row(case, t_value=float(t_value)) for case in benchmark_cases() for t_value in ALGO18_T_VALUES]
display(dataframe_result('algo18_wyckoff_and_kinematic_sanity', rows))


[K2 graph=2 t=0.20] finite_ok=True mean_free_norm=1.617e-09 eps_v_mean_norm=3.032e-08 lattice_changed=0.000e+00
[W/K] graph=2 t=0.20 c_ref=0.000e+00 orth_err=0.000e+00 tangent_vals=[3.566890416299137e-16, 6.80336863234654e-16] boundary_c=0.015006 grad_r=6.064e-06 grad_v=8.756e-06
[K2 graph=2 t=0.30] finite_ok=True mean_free_norm=2.982e-09 eps_v_mean_norm=7.068e-08 lattice_changed=0.000e+00
[W/K] graph=2 t=0.30 c_ref=0.000e+00 orth_err=0.000e+00 tangent_vals=[2.4949654569884916e-16, 3.616904006065359e-16] boundary_c=0.015006 grad_r=2.641e-05 grad_v=3.447e-05
[K2 graph=3 t=0.20] finite_ok=True mean_free_norm=2.043e-09 eps_v_mean_norm=1.863e-08 lattice_changed=0.000e+00
[W/K] graph=3 t=0.20 c_ref=0.000e+00 orth_err=1.192e-07 tangent_vals=[2.7287955195229416e-16, 4.253843665643031e-16] boundary_c=0.031317 grad_r=7.529e-04 grad_v=1.107e-03
[K2 graph=3 t=0.30] finite_ok=True mean_free_norm=3.754e-09 eps_v_mean_norm=3.183e-08 lattice_changed=0.000e+00
[W/K] graph=3 t=0.30 c_ref=0.000e+00 orth

,test,graph,space_group,t_value,constraint_ref,orthonormality_error_max,tangent_constraint_mean,boundary_constraint,grad_xi_r_norm,grad_xi_v_norm,finite_grad,state_mean_free_norm,renoise_finite_ok,renoise_mean_free_norm,renoise_lattice_changed_norm,PASS,status
0,algo18_wyckoff_and_kinematic_sanity,2,4,0.2,0.0,0.000000e+00,5.185130e-16,0.015006,6.064487e-06,0.000009,True,1.919971e-09,True,1.617293e-09,0.0,True,PASS
1,algo18_wyckoff_and_kinematic_sanity,2,4,0.3,0.0,0.000000e+00,3.055935e-16,0.015006,2.641284e-05,0.000034,True,1.766069e-09,True,2.981687e-09,0.0,True,PASS
2,algo18_wyckoff_and_kinematic_sanity,3,194,0.2,0.0,1.192093e-07,3.491320e-16,0.031317,7.529396e-04,0.001107,True,1.746230e-09,True,2.043081e-09,0.0,True,PASS
3,algo18_wyckoff_and_kinematic_sanity,3,194,0.3,0.0,1.192093e-07,1.331777e-16,0.031317,2.290409e-03,0.002878,True,1.317089e-09,True,3.754281e-09,0.0,True,PASS
4,algo18_wyckoff_and_kinematic_sanity,5,123,0.2,0.0,0.000000e+00,2.220446e-17,0.036015,8.291128e-07,0.000001,True,4.760002e-09,True,2.380001e-09,0.0,True,PASS
5,algo18_wyckoff_and_kinematic_sanity,5,123,0.3,0.0,0.000000e+00,2.220446e-17,0.036015,8.903563e-06,0.000011,True,3.530112e-09,True,8.528451e-09,0.0,True,PASS


In [ ]:
def _endpoint_eval_from_clean(case: GraphCase, clean_f: torch.Tensor, l_feature: torch.Tensor) -> dict[str, Any]:
    eval_after = evaluate_arrays(case, clean_f, l_feature, case.atomic_numbers)
    return {
        'frac_rmse': float(torus_rmse(clean_f, case.gt_frac).detach().item()),
        'match': bool(eval_after['match']),
        'valid': bool(eval_after['valid']),
        'rmse': eval_after['rmse'] if eval_after['rmse'] is not None else float('nan'),
        'min_pair_distance': eval_after['min_pair_distance'] if eval_after['min_pair_distance'] is not None else float('nan'),
        'metric_source': str(eval_after.get('metric_source')),
    }


def baseline_m0_row(case: GraphCase, *, t_value: float, variant: str) -> dict[str, Any]:
    try:
        fixture = build_fixture(case, variant)
        frame = fixture.get('operator_frame', fixture['frame'])
        state = algo18_state_from_noisy_gt(case, t_value=float(t_value), seed=SAMPLE_SEED)
        clean0 = algo18_clean_estimate(case, state)
        endpoint = _endpoint_eval_from_clean(case, clean0, state.l)
        c0 = float(frame.constraint.value(clean0).detach().item())
        print(f"[M0] graph={case.graph_id} variant={variant} t={t_value:.2f} c0={c0:.6f} frac_rmse={endpoint['frac_rmse']:.6f}", flush=True)
        return {
            'test': 'algo18_project_and_kernel',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'variant': variant,
            't_value': float(t_value),
            'M': 0,
            'control': 'baseline_M0',
            'c_before': c0,
            'c_after_project': c0,
            'c_after_renoise': c0,
            'frac_RMSE_after_renoise': endpoint['frac_rmse'],
            'match_after_renoise': endpoint['match'],
            'valid_after_renoise': endpoint['valid'],
            'RMSE_after_renoise': endpoint['rmse'],
            'min_pair_after_renoise': endpoint['min_pair_distance'],
            'metric_source': endpoint['metric_source'],
            'PASS': True,
            'status': 'INFO',
        }
    except Exception as exc:
        return error_row('algo18_project_and_kernel', case.graph_id, exc, 'ALGO18_PROJECT_KERNEL_ERROR', t_value=t_value, variant=variant, M=0)



def renoise_only_row(case: GraphCase, *, t_value: float, variant: str) -> dict[str, Any]:
    try:
        fixture = build_fixture(case, variant)
        frame = fixture.get('operator_frame', fixture['frame'])
        state = algo18_state_from_noisy_gt(case, t_value=float(t_value), seed=SAMPLE_SEED)
        batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
        t_graph, t_nodes = algo18_times(case, float(t_value))
        clean0 = algo18_clean_estimate(case, state)
        c_before = float(frame.constraint.value(clean0).detach().item())
        renoise = kldm_renoise_from_f0(
            model=runner.model,
            f0_star=clean0,
            l_t=state.l,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            debug=DEBUG_PRINTS,
            debug_prefix=f'[renoise-only graph={case.graph_id} {variant} t={t_value:.2f}]',
        )
        clean_after = kldm_clean_fractional_denoiser_Df(
            model=runner.model,
            f_t=renoise.f_t,
            v_t=renoise.v_t,
            l_t=state.l,
            a_t=state.h,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            variant=ALGO18_DENOISER_VARIANT,
        )
        endpoint = _endpoint_eval_from_clean(case, clean_after, state.l)
        c_after = float(frame.constraint.value(clean_after).detach().item())
        print(f"[renoise-only] graph={case.graph_id} variant={variant} t={t_value:.2f} c_before={c_before:.6f} c_after={c_after:.6f} frac_rmse={endpoint['frac_rmse']:.6f}", flush=True)
        return {
            'test': 'algo18_project_and_kernel',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'variant': variant,
            't_value': float(t_value),
            'M': 0,
            'control': 'renoise_only',
            'c_before': c_before,
            'c_after_project': c_before,
            'c_after_renoise': c_after,
            'renoise_only_mean_free_norm': float(renoise.mean_free_norm),
            'renoise_only_lattice_changed_norm': float(renoise.lattice_changed_norm),
            'frac_RMSE_after_renoise': endpoint['frac_rmse'],
            'match_after_renoise': endpoint['match'],
            'valid_after_renoise': endpoint['valid'],
            'RMSE_after_renoise': endpoint['rmse'],
            'min_pair_after_renoise': endpoint['min_pair_distance'],
            'metric_source': endpoint['metric_source'],
            'PASS': bool(renoise.finite_ok),
            'status': 'INFO' if bool(renoise.finite_ok) else 'WARN',
        }
    except Exception as exc:
        return error_row('algo18_project_and_kernel', case.graph_id, exc, 'ALGO18_PROJECT_KERNEL_ERROR', t_value=t_value, variant=variant, M=0, control='renoise_only')



def endpoint_row(case: GraphCase, *, t_value: float, variant: str, M: int) -> dict[str, Any]:
    try:
        fixture = build_fixture(case, variant)
        frame = fixture.get('operator_frame', fixture['frame'])
        state = algo18_state_from_noisy_gt(case, t_value=float(t_value), seed=SAMPLE_SEED)
        backend_state = algo18_to_backend_state(state)
        batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
        t_graph, t_nodes = algo18_times(case, float(t_value))
        config = algo18_config(repeats=max(int(M), 1), debug=DEBUG_PRINTS)

        grad = ppr_objective_gradient_sanity(
            model=runner.model,
            frame=frame,
            state=backend_state,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            config=config,
        )

        project = ppr_project_step(
            model=runner.model,
            frame=frame,
            state=backend_state,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            config=config,
            debug_prefix=f'[P2 graph={case.graph_id} {variant} t={t_value:.2f}]',
        )
        kernel = ppr_kernel_repeated(
            model=runner.model,
            frame=frame,
            state=backend_state,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            config=config,
            anchor_mode='soft',
        )
        hard_kernel = ppr_kernel_repeated(
            model=runner.model,
            frame=frame,
            state=backend_state,
            t_graph=t_graph,
            t_nodes=t_nodes,
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            config=Algorithm18Config(**{**algo18_config(repeats=1, debug=DEBUG_PRINTS).__dict__, 'repeats': 1}),
            anchor_mode='hard',
        )

        c_after_renoise = float('nan')
        frac_rmse_after = float('nan')
        valid_after = False
        rmse_after = float('nan')
        min_pair_after = float('nan')
        if kernel.final_state is not None:
            clean_after = kldm_clean_fractional_denoiser_Df(
                model=runner.model,
                f_t=kernel.final_state.f,
                v_t=kernel.final_state.v,
                l_t=kernel.final_state.l,
                a_t=kernel.final_state.h,
                t_graph=t_graph,
                t_nodes=t_nodes,
                node_index=batch_view.batch,
                edge_node_index=batch_view.edge_node_index,
                variant=config.denoiser_variant,
            )
            c_after_renoise = float(frame.constraint.value(clean_after).detach().item())
            endpoint = _endpoint_eval_from_clean(case, clean_after, kernel.final_state.l)
            frac_rmse_after = endpoint['frac_rmse']
            valid_after = endpoint['valid']
            rmse_after = endpoint['rmse']
            min_pair_after = endpoint['min_pair_distance']
        print(f"[R0/R1] graph={case.graph_id} variant={variant} t={t_value:.2f} M={M} c_before={project.metrics.c_before:.6f} c_after_project={project.metrics.c_after_project:.6f} c_after_renoise={c_after_renoise}", flush=True)
        return {
            'test': 'algo18_project_and_kernel',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'variant': variant,
            't_value': float(t_value),
            'M': int(M),
            'control': 'ppr',
            'grad_xi_r_norm': grad['grad_xi_r_norm'],
            'grad_xi_v_norm': grad['grad_xi_v_norm'],
            'c_before': float(project.metrics.c_before),
            'c_after_project': float(project.metrics.c_after_project),
            'c_after_renoise': c_after_renoise,
            'loss_before': float(project.metrics.loss_before),
            'loss_after': float(project.metrics.loss_after),
            'xi_r_norm': float(project.metrics.xi_r_norm),
            'xi_v_norm': float(project.metrics.xi_v_norm),
            'delta_f_rms': float(project.metrics.delta_f_rms),
            'delta_v_rms': float(project.metrics.delta_v_rms),
            'mean_free_after_project': float(project.metrics.mean_free_norm),
            'kernel_accepted': bool(kernel.accepted),
            'kernel_reject_reason': kernel.reject_reason,
            'hard_kernel_accepted': bool(hard_kernel.accepted),
            'hard_kernel_reject_reason': hard_kernel.reject_reason,
            'frac_RMSE_after_renoise': frac_rmse_after,
            'match_after_renoise': bool(endpoint['match']) if kernel.final_state is not None else False,
            'valid_after_renoise': bool(valid_after),
            'RMSE_after_renoise': rmse_after,
            'min_pair_after_renoise': min_pair_after,
            'metric_source': endpoint['metric_source'] if kernel.final_state is not None else 'sample_evaluation.evaluate_csp_reconstruction',
            'PASS': bool(project.success and grad['finite_grad']),
            'status': 'PASS' if bool(project.success and grad['finite_grad']) else 'WARN',
        }
    except Exception as exc:
        return error_row('algo18_project_and_kernel', case.graph_id, exc, 'ALGO18_PROJECT_KERNEL_ERROR', t_value=t_value, variant=variant, M=M)


rows = []
for case in benchmark_cases():
    for t_value in ALGO18_T_VALUES:
        rows.append(baseline_m0_row(case, t_value=float(t_value), variant='correctW'))
        rows.append(renoise_only_row(case, t_value=float(t_value), variant='correctW'))
        for variant in ALGO18_VARIANTS:
            rows.append(endpoint_row(case, t_value=float(t_value), variant=str(variant), M=1))
        rows.append(endpoint_row(case, t_value=float(t_value), variant='correctW', M=2))

df_algo18_raw = dataframe_result('algo18_project_and_kernel', rows)
display(df_algo18_raw)

summary_rows = []
for case in benchmark_cases():
    for t_value in ALGO18_T_VALUES:
        sub = df_algo18_raw[(df_algo18_raw['graph'] == case.graph_id) & (df_algo18_raw['t_value'] == float(t_value))]
        frame_ok = True
        frame_df = result_tables.get('algo18_frame_problem_audit', pd.DataFrame())
        if not frame_df.empty:
            matched = frame_df[(frame_df['graph'] == case.graph_id)]
            if not matched.empty:
                frame_ok = bool(matched.iloc[0].get('frame_valid_for_science', False))
        def _first(control=None, variant=None, M=None):
            cur = sub if control is None else sub[sub['control'] == control]
            if variant is not None:
                cur = cur[cur['variant'] == variant]
            if M is not None:
                cur = cur[cur['M'] == M]
            return None if cur.empty else cur.iloc[0]
        base = _first(None, variant='correctW', M=0)
        renoise = _first('renoise_only', variant='correctW', M=0)
        correct1 = _first(None, variant='correctW', M=1)
        wrong1 = _first('ppr', variant='wrongSG', M=1)
        correct2 = _first(None, variant='correctW', M=2)
        scientific_pass = False
        if frame_ok and renoise is not None and correct1 is not None and wrong1 is not None:
            c_before = float(correct1['c_before'])
            c_correct = float(correct1['c_after_renoise']) if pd.notna(correct1['c_after_renoise']) else float('inf')
            c_wrong = float(wrong1['c_after_renoise']) if pd.notna(wrong1['c_after_renoise']) else float('inf')
            c_renoise = float(renoise['c_after_renoise']) if pd.notna(renoise['c_after_renoise']) else float('inf')
            frac_correct = float(correct1['frac_RMSE_after_renoise']) if pd.notna(correct1['frac_RMSE_after_renoise']) else float('inf')
            frac_wrong = float(wrong1['frac_RMSE_after_renoise']) if pd.notna(wrong1['frac_RMSE_after_renoise']) else float('inf')
            scientific_pass = bool(
                c_correct <= c_before + 1.0e-8
                and (float(c_before) - c_correct) > (float(c_before) - c_wrong) + ALGO18_C_SCI_EPS
                and (float(c_before) - c_correct) > (float(c_before) - c_renoise) + ALGO18_C_SCI_EPS
                and frac_correct <= frac_wrong + ALGO18_FRAC_RMSE_TOL
                and frac_correct <= float(renoise.get('frac_RMSE_after_renoise', float('inf'))) + ALGO18_FRAC_RMSE_TOL
                and bool(correct1.get('valid_after_renoise', False))
            )
        c_before = None if correct1 is None else correct1['c_before']
        correct_c = None if correct1 is None else correct1['c_after_renoise']
        wrong_c = None if wrong1 is None else wrong1['c_after_renoise']
        renoise_c = None if renoise is None else renoise['c_after_renoise']
        correct_delta_c = None if (c_before is None or correct_c is None or pd.isna(correct_c)) else float(c_before) - float(correct_c)
        wrong_delta_c = None if (c_before is None or wrong_c is None or pd.isna(wrong_c)) else float(c_before) - float(wrong_c)
        renoise_delta_c = None if (c_before is None or renoise_c is None or pd.isna(renoise_c)) else float(c_before) - float(renoise_c)
        correct_frac = None if correct1 is None else correct1['frac_RMSE_after_renoise']
        wrong_frac = None if wrong1 is None else wrong1['frac_RMSE_after_renoise']
        renoise_frac = None if renoise is None else renoise['frac_RMSE_after_renoise']
        summary_rows.append({
            'test': 'algo18_scientific_summary',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'frame_valid_for_science': bool(frame_ok),
            'baseline_c': None if base is None else base['c_after_renoise'],
            'renoise_only_c': renoise_c,
            'correctW_M1_c': correct_c,
            'wrongSG_M1_c': wrong_c,
            'correctW_M2_c': None if correct2 is None else correct2['c_after_renoise'],
            'correct_delta_c': correct_delta_c,
            'wrong_delta_c': wrong_delta_c,
            'renoise_only_delta_c': renoise_delta_c,
            'correct_frac_rmse': correct_frac,
            'wrong_frac_rmse': wrong_frac,
            'renoise_only_frac_rmse': renoise_frac,
            'correct_match_after': (None if correct1 is None or pd.isna(correct1.get('match_after_renoise')) else bool(correct1.get('match_after_renoise'))),
            'wrong_match_after': (None if wrong1 is None or pd.isna(wrong1.get('match_after_renoise')) else bool(wrong1.get('match_after_renoise'))),
            'renoise_only_match_after': (None if renoise is None or pd.isna(renoise.get('match_after_renoise')) else bool(renoise.get('match_after_renoise'))),
            'metric_source': 'sample_evaluation.evaluate_csp_reconstruction',
            'correct_beats_wrong_delta': None if (correct_delta_c is None or wrong_delta_c is None) else bool(correct_delta_c > wrong_delta_c + ALGO18_C_SCI_EPS),
            'correct_beats_renoise_delta': None if (correct_delta_c is None or renoise_delta_c is None) else bool(correct_delta_c > renoise_delta_c + ALGO18_C_SCI_EPS),
            'correct_not_worse_frac_than_wrong': None if (correct_frac is None or wrong_frac is None or pd.isna(correct_frac) or pd.isna(wrong_frac)) else bool(float(correct_frac) <= float(wrong_frac) + ALGO18_FRAC_RMSE_TOL),
            'correct_not_worse_frac_than_renoise': None if (correct_frac is None or renoise_frac is None or pd.isna(correct_frac) or pd.isna(renoise_frac)) else bool(float(correct_frac) <= float(renoise_frac) + ALGO18_FRAC_RMSE_TOL),
            'scientific_PASS': bool(scientific_pass),
            'PASS': bool(scientific_pass),
            'status': 'PASS' if bool(scientific_pass) else ('WARN' if bool(frame_ok) else 'FRAME_FAIL'),
        })

display(dataframe_result('algo18_scientific_summary', summary_rows))


[P2 graph=2 wrongSG t=0.20] step=0 loss=0.061729 c=0.061729 noise_pen=0.000000 xi_r_norm=0.138515 xi_v_norm=0.138522 mean_free=5.206e-10
[P2 graph=2 wrongSG t=0.20] step=1 loss=0.061623 c=0.061615 noise_pen=0.000799 xi_r_norm=0.220686 xi_v_norm=0.212761 mean_free=2.561e-09
[P2 graph=2 wrongSG t=0.20] step=2 loss=0.061584 c=0.061564 noise_pen=0.001958 xi_r_norm=0.277780 xi_v_norm=0.261623 mean_free=1.630e-09
[P2 graph=2 wrongSG t=0.20] step=3 loss=0.061537 c=0.061507 noise_pen=0.003034 xi_r_norm=0.324893 xi_v_norm=0.305443 mean_free=3.985e-09
[P2 graph=2 wrongSG t=0.20] step=4 loss=0.061486 c=0.061444 noise_pen=0.004143 xi_r_norm=0.369419 xi_v_norm=0.348858 mean_free=4.374e-09
[P2 graph=2 wrongSG t=0.20] step=5 loss=0.061464 c=0.061410 noise_pen=0.005379 xi_r_norm=0.405889 xi_v_norm=0.384812 mean_free=1.491e-09
[algo18 kernel] repeat=0 start
[algo18 project r0] step=0 loss=0.061729 c=0.061729 noise_pen=0.000000 xi_r_norm=0.138515 xi_v_norm=0.138522 mean_free=5.206e-10
[algo18 project r0

,test,graph,PASS,status,error_type,error_message,traceback_tail,failure_category,t_value,variant,...,kernel_accepted,kernel_reject_reason,hard_kernel_accepted,hard_kernel_reject_reason,frac_RMSE_after_renoise,match_after_renoise,valid_after_renoise,RMSE_after_renoise,min_pair_after_renoise,metric_source
0,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.2,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.2,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.2,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,algo18_project_and_kernel,2,True,PASS,NaN,NaN,NaN,NaN,0.2,wrongSG,...,True,,True,,0.005269,True,True,0.021850,2.042779,sample_evaluation.evaluate_csp_reconstruction
4,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.2,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.3,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.3,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.3,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,algo18_project_and_kernel,2,True,PASS,NaN,NaN,NaN,NaN,0.3,wrongSG,...,True,,True,,0.005105,True,True,0.021117,2.007704,sample_evaluation.evaluate_csp_reconstruction
9,algo18_project_and_kernel,2,False,ERROR,RecursionError,maximum recursion depth exceeded,RecursionError: maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR,0.3,correctW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,test,graph,space_group,t_value,frame_valid_for_science,baseline_c,renoise_only_c,correctW_M1_c,wrongSG_M1_c,correctW_M2_c,...,wrong_match_after,renoise_only_match_after,metric_source,correct_beats_wrong_delta,correct_beats_renoise_delta,correct_not_worse_frac_than_wrong,correct_not_worse_frac_than_renoise,scientific_PASS,PASS,status
0,algo18_scientific_summary,2,4,0.2,False,None,NaN,None,0.061505,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,FRAME_FAIL
1,algo18_scientific_summary,2,4,0.3,False,None,NaN,None,0.061866,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,FRAME_FAIL
2,algo18_scientific_summary,3,194,0.2,False,None,NaN,None,0.055988,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,FRAME_FAIL
3,algo18_scientific_summary,3,194,0.3,False,None,NaN,None,0.055686,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,FRAME_FAIL
4,algo18_scientific_summary,5,123,0.2,True,None,NaN,None,0.106441,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,WARN
5,algo18_scientific_summary,5,123,0.3,True,None,NaN,None,0.106232,None,...,True,None,sample_evaluation.evaluate_csp_reconstruction,None,None,None,None,False,False,WARN


In [27]:
operator_summary_rows = []
contract_df = result_tables.get('algo18_diffcsppp_contract_audit', pd.DataFrame())
for case in benchmark_cases():
    operator_fixture = build_fixture(case, 'correctW')
    case_contract = contract_df[contract_df['graph'] == case.graph_id] if not contract_df.empty else pd.DataFrame()
    contract_ok = bool(not case_contract.empty and bool(case_contract['PASS'].all()))
    operator_frame = operator_fixture.get('operator_frame', operator_fixture['frame'])
    operator_frame_ok = bool(contract_ok)
    for t_value in ALGO18_T_VALUES:
        sub = df_algo18_raw[(df_algo18_raw['graph'] == case.graph_id) & (df_algo18_raw['t_value'] == float(t_value))]
        def _first(control, variant=None, M=None):
            cur = sub[sub['control'] == control]
            if variant is not None:
                cur = cur[cur['variant'] == variant]
            if M is not None:
                cur = cur[cur['M'] == M]
            return None if cur.empty else cur.iloc[0]
        base = _first('baseline_M0', variant='correctW', M=0)
        renoise = _first('renoise_only', variant='correctW', M=0)
        correct1 = _first('ppr', variant='correctW', M=1)
        wrong1 = _first('ppr', variant='wrongSG', M=1)
        correct2 = _first('ppr', variant='correctW', M=2)
        scientific_pass = False
        if operator_frame_ok and renoise is not None and correct1 is not None and wrong1 is not None:
            c_before = float(correct1['c_before'])
            c_correct = float(correct1['c_after_renoise']) if pd.notna(correct1['c_after_renoise']) else float('inf')
            c_wrong = float(wrong1['c_after_renoise']) if pd.notna(wrong1['c_after_renoise']) else float('inf')
            c_renoise = float(renoise['c_after_renoise']) if pd.notna(renoise['c_after_renoise']) else float('inf')
            frac_correct = float(correct1['frac_RMSE_after_renoise']) if pd.notna(correct1['frac_RMSE_after_renoise']) else float('inf')
            frac_wrong = float(wrong1['frac_RMSE_after_renoise']) if pd.notna(wrong1['frac_RMSE_after_renoise']) else float('inf')
            scientific_pass = bool(
                c_correct <= c_before + 1.0e-8
                and (float(c_before) - c_correct) > (float(c_before) - c_wrong) + ALGO18_C_SCI_EPS
                and (float(c_before) - c_correct) > (float(c_before) - c_renoise) + ALGO18_C_SCI_EPS
                and frac_correct <= frac_wrong + ALGO18_FRAC_RMSE_TOL
                and frac_correct <= float(renoise.get('frac_RMSE_after_renoise', float('inf'))) + ALGO18_FRAC_RMSE_TOL
                and bool(correct1.get('valid_after_renoise', False))
            )
        operator_summary_rows.append({
            'test': 'algo18_operator_scientific_summary',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'frame_valid_for_science': bool(operator_frame_ok),
            'baseline_c': None if base is None else base['c_after_renoise'],
            'renoise_only_c': None if renoise is None else renoise['c_after_renoise'],
            'correctW_M1_c': None if correct1 is None else correct1['c_after_renoise'],
            'wrongSG_M1_c': None if wrong1 is None else wrong1['c_after_renoise'],
            'correctW_M2_c': None if correct2 is None else correct2['c_after_renoise'],
            'scientific_PASS': bool(scientific_pass),
            'PASS': bool(scientific_pass),
            'status': 'PASS' if bool(scientific_pass) else ('WARN' if bool(operator_frame_ok) else 'FRAME_FAIL'),
        })

display(dataframe_result('algo18_operator_scientific_summary', operator_summary_rows))


,test,graph,space_group,t_value,frame_valid_for_science,baseline_c,renoise_only_c,correctW_M1_c,wrongSG_M1_c,correctW_M2_c,scientific_PASS,PASS,status
0,algo18_operator_scientific_summary,2,4,0.2,True,None,NaN,None,0.061505,None,False,False,WARN
1,algo18_operator_scientific_summary,2,4,0.3,True,None,NaN,None,0.061866,None,False,False,WARN
2,algo18_operator_scientific_summary,3,194,0.2,True,None,NaN,None,0.055988,None,False,False,WARN
3,algo18_operator_scientific_summary,3,194,0.3,True,None,NaN,None,0.055686,None,False,False,WARN
4,algo18_operator_scientific_summary,5,123,0.2,True,None,NaN,None,0.106441,None,False,False,WARN
5,algo18_operator_scientific_summary,5,123,0.3,True,None,NaN,None,0.106232,None,False,False,WARN


In [20]:
print('df_algo18_raw_shape=', getattr(df_algo18_raw, 'shape', None))
if isinstance(df_algo18_raw, pd.DataFrame) and not df_algo18_raw.empty:
    print('controls=', df_algo18_raw['control'].value_counts(dropna=False).to_dict())
    display(df_algo18_raw[['graph', 'control', 'variant', 'M', 'c_before', 'c_after_renoise']].head(12))


df_algo18_raw_shape= (30, 35)
controls= {nan: 18, 'renoise_only': 6, 'ppr': 6}


,graph,control,variant,M,c_before,c_after_renoise
0,2,NaN,correctW,0,NaN,NaN
1,2,renoise_only,correctW,0,NaN,NaN
2,2,NaN,correctW,1,NaN,NaN
3,2,ppr,wrongSG,1,0.061729,0.061505
4,2,NaN,correctW,2,NaN,NaN
5,2,NaN,correctW,0,NaN,NaN
6,2,renoise_only,correctW,0,NaN,NaN
7,2,NaN,correctW,1,NaN,NaN
8,2,ppr,wrongSG,1,0.061514,0.061866
9,2,NaN,correctW,2,NaN,NaN


In [21]:
error_rows = df_algo18_raw[df_algo18_raw['control'].isna()].copy()
if not error_rows.empty:
    display(error_rows[['graph', 'variant', 'M', 'error_type', 'error_message', 'failure_category']].head(12))


,graph,variant,M,error_type,error_message,failure_category
0,2,correctW,0,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
2,2,correctW,1,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
4,2,correctW,2,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
5,2,correctW,0,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
7,2,correctW,1,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
9,2,correctW,2,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
10,3,correctW,0,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
12,3,correctW,1,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
14,3,correctW,2,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR
15,3,correctW,0,RecursionError,maximum recursion depth exceeded,ALGO18_PROJECT_KERNEL_ERROR


In [28]:
if isinstance(df_algo18_raw, pd.DataFrame) and not df_algo18_raw.empty:
    diag = df_algo18_raw[df_algo18_raw['graph'] == 2][['t_value', 'control', 'variant', 'M', 'c_after_renoise']].copy()
    display(diag.sort_values(['t_value', 'control', 'variant', 'M']).head(20))


,t_value,control,variant,M,c_after_renoise
3,0.2,ppr,wrongSG,1,0.061505
1,0.2,renoise_only,correctW,0,NaN
0,0.2,NaN,correctW,0,NaN
2,0.2,NaN,correctW,1,NaN
4,0.2,NaN,correctW,2,NaN
8,0.3,ppr,wrongSG,1,0.061866
6,0.3,renoise_only,correctW,0,NaN
5,0.3,NaN,correctW,0,NaN
7,0.3,NaN,correctW,1,NaN
9,0.3,NaN,correctW,2,NaN
